# 53. FM52 → LiDAR odometry corruption (New College-motivated domain shift)

Fine-tune the **FM52 yaw-robust checkpoint**, preserving the original KAIST **one-IMU / two-LiDAR** topology and the correction-to-go + differentiable endpoint objective. This experiment changes **only LiDAR relative SE(3) measurements**: per-edge SO(3) registration noise, temporally correlated rotational drift, and optional per-window translation-scale distortion. No artificial IMU noise and no change to calibration targets or timing.

**Why these initial corruption magnitudes?** On New College, Ouster gyro versus LiDAR rotations disagree by approximately **0.307° median per ~0.1 s scan edge** (after a +60 ms alignment), and by **5.14° median over 20 edges**. Those are *cross-sensor residuals*, **not measured ICP noise**, and include IMU, temporal, and convention effects. We therefore start with a plausible corruption family, verify its actual severity on KAIST increments, and provide fixed-noise and curriculum settings. It is not a statistical fit to New College errors.

**Experiment plan**
1. Run invariants and corruption-severity checks, including a clean identity case and a paired-view check.
2. Run the short **stochastic diagnostic** (saved best candidate; original model/RNG restored afterward). Evaluate identical clean and noisy cases by fine/coarse and Z-yaw.
3. If successful, enable **full training**. Evaluate on both clean and fully corrupted validation windows and select checkpoints using a fixed weighted score. The source FM52 checkpoint remains unchanged.
4. Optionally run the **fixed yaw × injected-angle** comparison between FM52 and FM53, on matched clean/noisy cases. New College remains an external transfer test, not part of training or hyperparameter fitting.

**Dependency:** Install the modified `src/obscalib/augmentations/noise.py` containing `OdometryCorruptionAugmenter`, `OdometryCorruptionAugmentationConfig`, and `SE3OdometryCorruptionConfig` before running this notebook.

## 1. Imports and paths

In [ ]:
from pathlib import Path
from dataclasses import asdict, fields, is_dataclass, replace
from collections import defaultdict
import gc
import math
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "obscalib").is_dir():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Run from the nn-obs-calib repository root (must contain src/obscalib).")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from kaist_dataset.dataset import KAISTCalibrationDataset, KAISTRigCalibration, build_kaist_sequence_specs, make_kaist_dataloader
from obscalib.augmentations.config import AugmentationConfig, CalibrationEventConfig, CommonBodyFrameRandomizationConfig, FrameRandomizationConfig, NoiseAugmentationConfig, PerturbationMagnitudeConfig, PriorPerturbationConfig, SamplingRateAugmentationConfig, SensorTopologyAugmentationConfig, TrueCalibrationAugmentationConfig
from obscalib.augmentations.pipeline import AugmentationPipeline
from obscalib.augmentations.noise import OdometryCorruptionAugmenter, OdometryCorruptionAugmentationConfig, SE3OdometryCorruptionConfig
from obscalib.data.structures import GeometryType
from obscalib.augmentations.common_body_frame import CommonBodyFrameRandomizer
from obscalib.augmentations import PerturbationDistribution
from obscalib.augmentations.prior_perturbation import CalibrationPriorPerturber, apply_prior_perturbation
from obscalib.calibration.update import CalibrationUpdater
from obscalib.config import CalibrationFlowIntegratorConfig, CalibrationFlowLossConfig, CalibrationPoolerConfig, MLPConfig, PooledCalibrationFlowConfig, PooledCalibrationHeadConfig, PooledCalibrationModelConfig, TransformerConfig, WindowingConfig
from obscalib.data.collation import collate_windows
from obscalib.data.device import window_batch_to
from obscalib.data.structures import WindowBatch, TokenBatch
from obscalib.flow_matching.loss import calibration_velocity_loss
from obscalib.flow_matching.calibration_path import make_correction_to_go_flow_batch
from obscalib.geometry.lie import se3_exp, se3_log
from obscalib.geometry.processing import GeometryProcessor
from obscalib.models.pooled_calibration_model import PooledCalibrationModel
from obscalib.models.pooled_calibration_flow_model import PooledCalibrationFlowModel
from obscalib.pipeline.window_step import WindowStep
from obscalib.tokenization.tokenizer import Tokenizer
from obscalib.training.checkpoint import load_training_checkpoint, save_training_checkpoint

print("ROOT", ROOT, "| PyTorch", torch.__version__, "| CUDA", torch.cuda.is_available())


## 2. FM52 initialization and independent odometry-corruption settings

In [ ]:
RUN_NAME = "kaist_pooled_calibration_fm.10_lidar_odometry_noise"
SOURCE_FM_CHECKPOINT = ROOT / "runs" / "kaist_pooled_calibration_fm.9_signed_z_yaw" / "best.pt"
RUN_DIR = ROOT / "runs" / RUN_NAME
BEST_CHECKPOINT_PATH, LAST_CHECKPOINT_PATH = RUN_DIR / "best.pt", RUN_DIR / "last.pt"

SEED = 12345
TRAIN_AUG_SEED, VAL_AUG_SEED, TEST_AUG_SEED = 23456, 34567, 45678
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
CALIBRATION_KEYS = ("imu", "lidar", "lidar_right")
MEASUREMENT_DIM = 6
WINDOW_DURATION_S, WINDOW_STRIDE_S = 5.0, 5.0
MAX_SAMPLES_PER_SENSOR = 700
BATCH_SIZE, NUM_WORKERS = 8, 0
HEAD_LEARNING_RATE, BACKBONE_LEARNING_RATE = 2e-5, 2e-6
WEIGHT_DECAY, GRADIENT_CLIP_NORM = 1e-4, 1.0
GRADIENT_ACCUMULATION_STEPS = 4
FLOW_PATH_NUM_POINTS = 9
TRAIN_INTEGRATION_STEPS, TRAIN_INTEGRATION_METHOD = 4, "euler"
FM_INTEGRATION_STEPS, FM_INTEGRATION_METHOD = 16, "midpoint"
ABLATION_STEPS = (1, 2, 4, 8, 16, 32)
VELOCITY_LOSS_WEIGHT, ENDPOINT_LOSS_WEIGHT = 1.0, 0.2

FINE_PRIOR_PROBABILITY = 0.50
FINE_ROTATION_MAX_RAD, FINE_ROTATION_SCALE_RAD = np.deg2rad(60.), np.deg2rad(30.)
FINE_TIME_OFFSET_MAX_S, FINE_TIME_OFFSET_SCALE_S = 0.5, 0.25
COARSE_TIME_OFFSET_MAX_S, COARSE_TIME_OFFSET_SCALE_S = 0.5, 0.25
STUDENT_T_DEGREES_OF_FREEDOM = 3
TRUE_ROTATION_MAX_RAD, TRUE_TIME_OFFSET_MAX_S = np.deg2rad(180.), 0.030

# Original topology, FM52 signed yaw robustness preserved. Use the FULL yaw range immediately.
TRAIN_PAIRED_VIEWS = True
ROTATED_VIEW_WEIGHT, EQUIVARIANCE_LOSS_WEIGHT = 1.0, 0.05
FULL_YAW_PROBABILITY = 0.90
USE_YAW_CURRICULUM = False
FULL_FIXED_YAW_MAX_DEG = 180.0
FULL_YAW_SCHEDULE = ((1, 180.0),)
VAL_YAW_PROBABILITY, VAL_YAW_MAX_DEG = 1.0, 180.0
YAW_AXES = (False, False, True)

# --- New College-motivated LiDAR odometry corruption, NOT fitted ICP noise ---
# New College residuals: ~0.307 deg/edge, ~5.14 deg/20 edges; do not interpret
# either residual as a ground-truth noise standard deviation.
ODOM_ROTATION_NOISE_STD_DEG = 0.18          # Per-axis 1-sigma angular registration error per edge.
ODOM_ROTATION_BIAS_RATE_STD_DEG_S = 1.0     # Per-axis stationary OU bias rate, deg/s.
ODOM_ROTATION_BIAS_CORRELATION_S = 2.0     # Approximately 20 scans at 10 Hz.
ODOM_TRANSLATION_SCALE_MIN = 0.85         # Per-window scale applied to all LiDAR translation edges.
ODOM_TRANSLATION_SCALE_MAX = 1.15
ODOM_ENABLE_ROTATION = True               # Independent ablation flags, without changing the model.
ODOM_ENABLE_TRANSLATION_SCALE = True
ODOM_CORRUPTION_PROBABILITY = 0.70        # Window-level: 70% both LiDAR streams corrupted; 30% fully clean.
# Scale both rotational stds and distance of translation bounds from 1 by intensity.
# intensity=1 => settings above. intensity=0 => no corruption even if selected.

# --- Stochastic preflight (NO optimizer/checkpoint side effects) ---
RUN_STOCHASTIC_CHECK = True
STOCHASTIC_STEPS, STOCHASTIC_REPORT_EVERY = 200, 10
STOCHASTIC_WINDOWS = 16
STOCHASTIC_TRAIN_REGIME = "mixed"              # "fine", "coarse", "mixed"
STOCHASTIC_COARSE_MAX_DEG = 179.0
STOCHASTIC_FINE_LOSS_WEIGHT, STOCHASTIC_COARSE_LOSS_WEIGHT = 0.50, 0.50
STOCHASTIC_HEAD_LR, STOCHASTIC_BACKBONE_LR = 5e-5, 5e-6
STOCHASTIC_YAW_MAX_DEG, STOCHASTIC_YAW_PROBABILITY = 180.0, 0.90
STOCHASTIC_PAIRED_VIEWS = True
STOCHASTIC_SAVE_CANDIDATE = True
STOCHASTIC_NOISE_SCHEDULE = "fixed"           # "fixed" or "ramp"
STOCHASTIC_NOISE_INTENSITY = 1.0             # 0.0 = clean, 1.0 = defaults, 1.5 = stronger.
STOCHASTIC_RAMP_START_INTENSITY = 0.20
STOCHASTIC_RAMP_UPDATES = 100
STOCHASTIC_SEED, STOCHASTIC_EVAL_SEED = TRAIN_AUG_SEED + 200000, VAL_AUG_SEED + 300000

# --- Full run, independent from stochastic preflight ---
RUN_FULL_TRAINING = False                      # Enable after checking stochastic results.
RESUME_FULL_TRAINING = False                   # Resumes FM53 LAST.pt and optimizer only.
TRAIN_EPOCHS = 25
FULL_COARSE_MAX_DEG = 179.0
FULL_TRAIN_REGIME = "mixed"
FULL_FINE_LOSS_WEIGHT, FULL_COARSE_LOSS_WEIGHT = 0.50, 0.50
FULL_NOISE_SCHEDULE = "fixed"                 # "fixed" or "ramp"
FULL_NOISE_INTENSITY = 1.0
FULL_RAMP_START_INTENSITY = 0.20
FULL_RAMP_EPOCHS = 5
VAL_NOISY_WEIGHT = 0.70                         # Score=0.7*noisy_loss+0.3*clean_loss; noise strength fixed.
MAX_TRAIN_BATCHES_PER_EPOCH, MAX_VAL_BATCHES_PER_EPOCH = None, None
LR_REDUCE_STEPS, LR_REDUCE_FACTOR = 8, 0.5
REQUIRE_PRECHECK_PASS_FOR_FULL_TRAINING = False

FULL_OBJECTIVE = dict(
    mode="mixed", off_path_probability=0.50, identity_probability=0.15,
    max_time=0.975, rotation_noise_std_rad=np.deg2rad(8.0),
    translation_noise_std_m=0.02, tau_noise_std_s=0.02,
    velocity_weight=VELOCITY_LOSS_WEIGHT, endpoint_weight=ENDPOINT_LOSS_WEIGHT,
    randomized_integration=True, randomized_query_times=True,
    on_policy_fraction=0.5, on_policy_steps=8,
    on_policy_max_time=0.875, on_policy_method="midpoint",
)
STOCHASTIC_OBJECTIVE = dict(FULL_OBJECTIVE)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == "cuda": torch.cuda.manual_seed_all(SEED)
for maximum in (STOCHASTIC_COARSE_MAX_DEG, FULL_COARSE_MAX_DEG):
    assert 60.0 <= maximum < 180.0
assert STOCHASTIC_NOISE_SCHEDULE in ("fixed", "ramp")
assert FULL_NOISE_SCHEDULE in ("fixed", "ramp")
assert 0.0 <= ODOM_CORRUPTION_PROBABILITY <= 1.0 and 0.0 <= VAL_NOISY_WEIGHT <= 1.0
assert SOURCE_FM_CHECKPOINT.is_file(), f"FM52 source checkpoint not found: {SOURCE_FM_CHECKPOINT}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("FM52 source:", SOURCE_FM_CHECKPOINT, "| run:", RUN_DIR)
print("Stochastic noise:", STOCHASTIC_NOISE_SCHEDULE, STOCHASTIC_NOISE_INTENSITY, "| Full training enabled:", RUN_FULL_TRAINING)


## 3. KAIST data (original three-sensor topology, unchanged)

In [ ]:
KAIST_DATASET_ROOT = Path("/mnt/d/Downloads/MobRobLab/KAISTDataset")
LIDAR_POSE_ROOT = Path("/home/camel/Skoltech/phd_proposal/data/KAISTDataset")
TRAIN_NAMES = ["Urban08", "Urban12", "Urban13", "Urban14", "Urban15", "Urban16"]
VAL_NAMES = ["Urban26", "Urban28"]
TEST_NAMES = ["Urban38", "Urban39"]
ALL_SEQUENCE_NAMES = TRAIN_NAMES + VAL_NAMES + TEST_NAMES
assert len(set(ALL_SEQUENCE_NAMES)) == len(ALL_SEQUENCE_NAMES)

T_B_L = np.array([[-0.514066, -0.702201, -0.492595, -0.440699], [0.486485, -0.711672, 0.506809, 0.397052], [-0.706447, 0.0208933, 0.707457, 1.90953], [0., 0., 0., 1.]], dtype=float)
T_B_L_right = np.array([[-0.512152, 0.699241, -0.498761, -0.449885], [-0.494811, -0.714859, -0.494104, -0.416713], [-0.702041, -0.0062642, 0.712109, 1.91294], [0., 0., 0., 1.]], dtype=float)
T_I_B = np.array([[1., 0., 0., -0.07], [0., 1., 0., 0.], [0., 0., 1., 1.7], [0., 0., 0., 1.]], dtype=float)
rig_calibration = KAISTRigCalibration(T_B_I=np.linalg.inv(T_I_B), T_B_L=T_B_L, tau_imu_s=0.0, tau_lidar_s=0.0, T_B_L_right=T_B_L_right, tau_lidar_right_s=0.0)
windowing_config = WindowingConfig(window_duration_s=WINDOW_DURATION_S, window_stride_s=WINDOW_STRIDE_S, max_samples_per_sensor=MAX_SAMPLES_PER_SENSOR)
all_specs = build_kaist_sequence_specs(sequence_names=ALL_SEQUENCE_NAMES, kaist_dataset_root=KAIST_DATASET_ROOT, lidar_pose_root=LIDAR_POSE_ROOT, lidar_right_pose_filename="lidar_map_poses_vlp_right.csv")
for spec in all_specs:
    if spec.lidar_right_pose_csv is None or not spec.lidar_right_pose_csv.is_file():
        raise FileNotFoundError(f"Right-LiDAR pose CSV required by 002.24 architecture: {spec.name}: {spec.lidar_right_pose_csv}")
spec_by_name = {spec.name: spec for spec in all_specs}

def make_dataset(names):
    return KAISTCalibrationDataset([spec_by_name[name] for name in names], rig_calibration, windowing_config, measurement_dtype=DTYPE, source_timestamp_dtype=torch.float64, window_timestamp_dtype=DTYPE)

train_dataset, val_dataset, test_dataset = (make_dataset(names) for names in (TRAIN_NAMES, VAL_NAMES, TEST_NAMES))
for name, ds in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    if tuple(ds.calibration_keys) != CALIBRATION_KEYS:
        raise AssertionError(f"Unexpected calibration keys for {name}: {ds.calibration_keys}")
    print(name, len(ds), "windows")

train_shuffle_generator = torch.Generator().manual_seed(SEED)
train_loader = make_kaist_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS, generator=train_shuffle_generator)
val_loader = make_kaist_dataloader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)


## 4. Physical truth, calibration priors, yaw views, and LiDAR-only measurement corruption

In [ ]:
true_calibration_config = AugmentationConfig(
    sensor_topology=SensorTopologyAugmentationConfig(enabled=False),
    sampling_rate=SamplingRateAugmentationConfig(enabled=False),
    frame_randomization=FrameRandomizationConfig(enabled=False),
    common_body_frame=CommonBodyFrameRandomizationConfig(enabled=False, probability=0.0),
    true_calibration=TrueCalibrationAugmentationConfig(
        enabled=True, probability=1.0, rotation_probability=1.0, translation_probability=0.0, time_offset_probability=1.0,
        rotation=PerturbationMagnitudeConfig(maximum=TRUE_ROTATION_MAX_RAD),
        time_offset=PerturbationMagnitudeConfig(maximum=TRUE_TIME_OFFSET_MAX_S),
    ),
    prior_perturbation=PriorPerturbationConfig(enabled=False),
    calibration_event=CalibrationEventConfig(enabled=False),
    noise=NoiseAugmentationConfig(enabled=False),
)

fine_prior_config = PriorPerturbationConfig(
    enabled=True, probability=1.0,
    rotation=PerturbationMagnitudeConfig(maximum=FINE_ROTATION_MAX_RAD, distribution=PerturbationDistribution.TRUNCATED_STUDENT_T, scale=FINE_ROTATION_SCALE_RAD, degrees_of_freedom=STUDENT_T_DEGREES_OF_FREEDOM),
    translation=None,
    time_offset=PerturbationMagnitudeConfig(maximum=FINE_TIME_OFFSET_MAX_S, distribution=PerturbationDistribution.TRUNCATED_STUDENT_T, scale=FINE_TIME_OFFSET_SCALE_S, degrees_of_freedom=STUDENT_T_DEGREES_OF_FREEDOM),
    one_calibration_per_sample=True, one_rotation_axis_per_sample=False, one_perturbation_type_per_sample=False,
)
coarse_prior_config = PriorPerturbationConfig(
    enabled=True, probability=1.0,
    rotation=PerturbationMagnitudeConfig(maximum=np.deg2rad(FULL_COARSE_MAX_DEG), distribution=PerturbationDistribution.UNIFORM),
    translation=None,
    time_offset=PerturbationMagnitudeConfig(maximum=COARSE_TIME_OFFSET_MAX_S, distribution=PerturbationDistribution.UNIFORM, scale=COARSE_TIME_OFFSET_SCALE_S, degrees_of_freedom=STUDENT_T_DEGREES_OF_FREEDOM),
    one_calibration_per_sample=True, one_rotation_axis_per_sample=False, one_perturbation_type_per_sample=False,
)
fine_prior_perturber = CalibrationPriorPerturber(fine_prior_config)
coarse_prior_perturber = CalibrationPriorPerturber(coarse_prior_config)

def augmentation_generator(seed: int):
    return torch.Generator(device=DEVICE.type).manual_seed(int(seed))

def clone_with_current_calibration(window: WindowBatch, new_calibration):
    return WindowBatch(streams=dict(window.streams), current_calibration=new_calibration, metadata=dict(window.metadata), targets=None if window.targets is None else dict(window.targets), dataset_indices=window.dataset_indices)

# Probability that each calibration receives the ONE prior fault in a physical window.
# Keep original KAIST unchanged; emphasize the remaining real LiDAR in 2-IMU/1-LiDAR cases.
FAULT_KEY_WEIGHTS = {
    frozenset(("imu", "lidar", "lidar_right")): {"imu": 1.0, "lidar": 1.0, "lidar_right": 1.0},
    frozenset(("imu", "imu_synth", "lidar_right")): {"imu": 0.20, "imu_synth": 0.20, "lidar_right": 0.60},
    frozenset(("imu", "lidar", "imu_synth")): {"imu": 0.20, "lidar": 0.60, "imu_synth": 0.20},
}

FAULT_KEY_WEIGHTS = {frozenset(CALIBRATION_KEYS): {key: 1.0 for key in CALIBRATION_KEYS}}
def sample_fine_coarse_deltas(true_states, generator: torch.Generator, regime: str = "mixed"):
    if regime not in {"mixed", "fine", "coarse"}:
        raise ValueError(regime)

    keys = tuple(true_states)
    probability_by_key = FAULT_KEY_WEIGHTS.get(frozenset(keys))
    if probability_by_key is None:
        raise ValueError(f"No fault-selection weights for calibration keys {keys}")

    template = next(iter(true_states.values())).transform
    n = len(template)
    weights = torch.tensor([probability_by_key[key] for key in keys], device=template.device, dtype=torch.float32)
    weights = weights / weights.sum()
    selected_key_indices = torch.multinomial(weights, n, replacement=True, generator=generator)

    if regime == "mixed":
        fine_mask = torch.rand(n, device=template.device, generator=generator) < FINE_PRIOR_PROBABILITY
    else:
        fine_mask = torch.full((n,), regime == "fine", device=template.device, dtype=torch.bool)

    # Reuse the existing perturbers on single-key dictionaries. This preserves their
    # exact Student-t/uniform SO(3) and tau distributions while controlling key choice.
    xi, tau = {}, {}
    for key_index, key in enumerate(keys):
        singleton = {key: true_states[key]}
        if regime != "coarse":
            _, fine_xi, fine_tau = fine_prior_perturber(singleton, generator=generator)
        if regime != "fine":
            _, coarse_xi, coarse_tau = coarse_prior_perturber(singleton, generator=generator)

        if regime == "fine":
            chosen_xi, chosen_tau = fine_xi[key], fine_tau[key]
        elif regime == "coarse":
            chosen_xi, chosen_tau = coarse_xi[key], coarse_tau[key]
        else:
            chosen_xi = torch.where(fine_mask[:, None], fine_xi[key], coarse_xi[key])
            chosen_tau = torch.where(fine_mask[:, None], fine_tau[key], coarse_tau[key])

        selected = (selected_key_indices == key_index)[:, None]
        xi[key] = torch.where(selected, chosen_xi, torch.zeros_like(chosen_xi))
        tau[key] = torch.where(selected, chosen_tau, torch.zeros_like(chosen_tau))

    return xi, tau, fine_mask

def correction_targets(window: WindowBatch):
    if window.targets is None:
        raise ValueError("Expected truth targets after topology/true-calibration augmentation")
    targets_xi, targets_tau = {}, {}
    for key, state in window.current_calibration.items():
        target = window.targets[key]
        if target.next_transform is None or target.next_time_offset is None:
            raise ValueError(f"Missing correction truth for {key}")
        targets_xi[key] = se3_log(target.next_transform @ torch.linalg.inv(state.transform))
        targets_tau[key] = target.next_time_offset - state.time_offset
    return targets_xi, targets_tau



from obscalib.augmentations.common_body_frame import CommonBodyFrameRandomizer
physical_augmenter = AugmentationPipeline(true_calibration_config)


@torch.no_grad()
def augment_original_window(base_window: WindowBatch, generator: torch.Generator, *, regime="mixed"):
    """Physical truth + exactly one calibration prior. Never replaces sensor streams."""
    physical = physical_augmenter(base_window, generator=generator).augmented_window
    assert tuple(physical.current_calibration) == CALIBRATION_KEYS
    xi, tau, _ = sample_fine_coarse_deltas(physical.current_calibration, generator, regime=regime)
    return clone_with_current_calibration(physical, apply_prior_perturbation(physical.current_calibration, xi, tau))


def make_yaw_randomizer(max_deg, probability=1.0):
    """Notebook 39 common-body-frame augmentation: only rotations about +Z, signed."""
    assert 0. <= probability <= 1.
    assert max_deg is None or 0. <= max_deg <= 180.
    rotation = None if max_deg is None else PerturbationMagnitudeConfig(maximum=np.deg2rad(float(max_deg)), distribution=PerturbationDistribution.UNIFORM)
    config = CommonBodyFrameRandomizationConfig(enabled=True, probability=probability, rotation_axes=YAW_AXES, rotation=rotation)
    return CommonBodyFrameRandomizer(config)


def training_yaw_cap(epoch):
    if not USE_YAW_CURRICULUM:
        return FULL_FIXED_YAW_MAX_DEG
    value = FULL_YAW_SCHEDULE[0][1]
    for first_epoch, maximum in FULL_YAW_SCHEDULE:
        if epoch < first_epoch: break
        value = maximum
    return value


def rotate_yaw_view(window, generator, yaw_randomizer):
    """Rotate both the CURRENT and TARGET calibrations, preserving raw sensor data."""
    rotated, _, H = yaw_randomizer(window, generator=generator)
    assert set(rotated.current_calibration) == set(window.current_calibration)
    return rotated, H


def rotate_twists(xi, H):
    """Adjoint for a rotation-only common-frame transform, twist order [omega, v]."""
    R = H[:, :3, :3]
    return torch.cat((torch.bmm(R, xi[:, :3, None]).squeeze(-1), torch.bmm(R, xi[:, 3:, None]).squeeze(-1)), dim=-1)


def rotate_window_fixed_yaw(window, yaw_degrees):
    """Deterministic counterpart of CommonBodyFrameRandomizer for evaluation."""
    sample = next(iter(window.current_calibration.values())).transform
    yaw = sample.new_tensor(np.deg2rad(yaw_degrees))
    co, si = torch.cos(yaw), torch.sin(yaw)
    H = torch.eye(4, device=sample.device, dtype=sample.dtype).unsqueeze(0).repeat(len(sample), 1, 1)
    H[:, 0, 0], H[:, 0, 1] = co, -si
    H[:, 1, 0], H[:, 1, 1] = si, co
    current = {key: replace(state, transform=H @ state.transform) for key, state in window.current_calibration.items()}
    targets = None if window.targets is None else {
        key: replace(target, next_transform=H @ target.next_transform)
        for key, target in window.targets.items()
    }
    return WindowBatch(streams=dict(window.streams), metadata=dict(window.metadata), current_calibration=current, targets=targets, dataset_indices=window.dataset_indices), H


print("Original topology only", CALIBRATION_KEYS, "| Fine and coarse priors unchanged.")


### 4a. Odometry-corruption profiles and clean/noisy window partitioning

In [ ]:
# Require the patched noise.py containing OdometryCorruptionAugmenter.
# This is an augmentation of sensor measurements; calibration targets remain fixed.

LIDAR_STREAM_KEYS = tuple(key for key in train_dataset[0].streams if key.startswith("lidar"))
assert LIDAR_STREAM_KEYS and set(LIDAR_STREAM_KEYS) == {"lidar", "lidar_right"}, LIDAR_STREAM_KEYS
LIDAR_MEASUREMENT_TYPES = tuple(dict.fromkeys(train_dataset[0].metadata[key].measurement_type for key in LIDAR_STREAM_KEYS))
for key in LIDAR_STREAM_KEYS:
    assert train_dataset[0].metadata[key].geometry_type == GeometryType.SE3, f"Expected SE3 odometry for {key}"
for key, metadata in train_dataset[0].metadata.items():
    if key not in LIDAR_STREAM_KEYS and metadata.measurement_type in LIDAR_MEASUREMENT_TYPES:
        raise RuntimeError(f"Non-LiDAR {key} shares LiDAR measurement type; type-based noise selection would be unsafe")


def noise_config_dict():
    """Comparable immutable configuration fingerprint used for checkpoint resume."""
    return dict(probability=ODOM_CORRUPTION_PROBABILITY, white_deg=ODOM_ROTATION_NOISE_STD_DEG,
                bias_deg_s=ODOM_ROTATION_BIAS_RATE_STD_DEG_S, correlation_s=ODOM_ROTATION_BIAS_CORRELATION_S,
                scale_min=ODOM_TRANSLATION_SCALE_MIN, scale_max=ODOM_TRANSLATION_SCALE_MAX,
                enable_rotation=ODOM_ENABLE_ROTATION, enable_translation=ODOM_ENABLE_TRANSLATION_SCALE,
                lidar_types=tuple(str(v) for v in LIDAR_MEASUREMENT_TYPES))


def odometry_noise_augmenter(intensity=1.0):
    """Single corruption profile, applied to all selected LiDAR streams in the window.

    With intensity=1, expected per-edge rotation noise is comparable to the measured
    ~0.307 degree New College gyro/LiDAR disagreement (not equal by construction).
    The nominal profile is deliberately a hypothesis to validate, not an ICP noise fit.
    """
    intensity = float(intensity)
    if not 0.0 <= intensity <= 3.0:
        raise ValueError("Noise intensity must be between 0 and 3")
    rotation_std = ODOM_ROTATION_NOISE_STD_DEG * intensity if ODOM_ENABLE_ROTATION else 0.0
    bias_std = ODOM_ROTATION_BIAS_RATE_STD_DEG_S * intensity if ODOM_ENABLE_ROTATION else 0.0
    scale_min = 1.0 + intensity * (ODOM_TRANSLATION_SCALE_MIN - 1.0) if ODOM_ENABLE_TRANSLATION_SCALE else 1.0
    scale_max = 1.0 + intensity * (ODOM_TRANSLATION_SCALE_MAX - 1.0) if ODOM_ENABLE_TRANSLATION_SCALE else 1.0
    cfg = SE3OdometryCorruptionConfig(probability=1.0, rotation_noise_std_deg=rotation_std,
                                      rotation_bias_rate_std_deg_s=bias_std, rotation_bias_correlation_time_s=ODOM_ROTATION_BIAS_CORRELATION_S,
                                      translation_scale_min=scale_min, translation_scale_max=scale_max)
    return OdometryCorruptionAugmenter(OdometryCorruptionAugmentationConfig(enabled=True, se3_by_type={t: cfg for t in LIDAR_MEASUREMENT_TYPES}))


def noise_intensity(progress, *, schedule, target, ramp_start, ramp_length):
    """Saturated linear curriculum, progress starts at 1 (step or epoch)."""
    if schedule == "fixed": return float(target)
    if schedule != "ramp": raise ValueError(schedule)
    fraction = min(1.0, max(0.0, (progress - 1) / max(1, ramp_length - 1)))
    return float(ramp_start + fraction * (target - ramp_start))


def noise_subgroups(window, generator, probability):
    """Window-level random clean/corrupt split shared by both LiDAR streams.

    No batch member is duplicated. This achieves 30% fully clean windows when
    probability=0.70, unlike independent per-LiDAR Bernoulli decisions.
    """
    if probability <= 0.0: return [("clean", window)]
    if probability >= 1.0: return [("corrupt", window)]
    n = len(next(iter(window.current_calibration.values())).transform)
    device = next(iter(window.current_calibration.values())).transform.device
    select = torch.rand(n, generator=generator, device=device) < probability
    return [(name, topology_slice_batch(window, indices))
            for name, indices in (("corrupt", select.nonzero(as_tuple=True)[0]),
                                  ("clean", (~select).nonzero(as_tuple=True)[0])) if indices.numel()]


def corrupt_lidar_window(window, generator, *, noise_mode="corrupt", augmenter=None):
    """Add measurement errors AFTER physical calibration faults and BEFORE tokenization."""
    if noise_mode == "clean": return window, {}
    if noise_mode != "corrupt": raise ValueError(noise_mode)
    if augmenter is None: augmenter = odometry_noise_augmenter(1.0)
    noisy, diagnostics = augmenter(window, generator=generator)
    assert set(diagnostics) == set(LIDAR_STREAM_KEYS), f"Unexpected corrupted streams: {tuple(diagnostics)}"
    return noisy, diagnostics


print("Odometry targets:", LIDAR_STREAM_KEYS, "| metadata types:", LIDAR_MEASUREMENT_TYPES)
print("Corruption profile:", noise_config_dict())


## 5. Training utilities and FM51 model

In [ ]:
def topology_slice_batch(window: WindowBatch, positions: torch.Tensor) -> WindowBatch:
    n = len(next(iter(window.current_calibration.values())).transform)
    if positions.numel() == 0:
        raise ValueError("Cannot create an empty topology group")
    positions_cpu = positions.detach().cpu().tolist()
    original_indices = window.dataset_indices
    if isinstance(original_indices, torch.Tensor):
        selected_indices = original_indices.index_select(0, positions.to(device=original_indices.device))
    elif original_indices is not None and hasattr(original_indices, "__len__") and len(original_indices) == n:
        selected = [original_indices[i] for i in positions_cpu]
        selected_indices = tuple(selected) if isinstance(original_indices, tuple) else selected
    else:
        selected_indices = original_indices

    def select(item):
        if isinstance(item, torch.Tensor) and item.ndim > 0 and item.shape[0] == n:
            return item.index_select(0, positions.to(device=item.device))
        if is_dataclass(item) and not isinstance(item, type):
            updates = {field.name: (selected_indices if isinstance(item, WindowBatch) and field.name == "dataset_indices" else select(getattr(item, field.name))) for field in fields(item) if field.init}
            return replace(item, **updates)
        if isinstance(item, dict):
            return {key: select(value) for key, value in item.items()}
        return item

    return select(window)


In [ ]:
model_config = PooledCalibrationModelConfig(
    transformer=TransformerConfig(input_dim=8, d_model=128, n_heads=8, num_layers=12, dim_feedforward=512, dropout=0.0, num_summary_tokens=1, activation="gelu"),
    pooler=CalibrationPoolerConfig(n_heads=8, dropout=0.0, calibration_context_dim=0),
    mlp=MLPConfig(hidden_dims=(256, 128), output_dim=128, activation="gelu", dropout=0.0),
    head=PooledCalibrationHeadConfig(calibration_context_dim=0, hidden_dims=(128, 64), activation="gelu", dropout=0.0),
)
flow_config = PooledCalibrationFlowConfig(
    backbone=model_config,
    integrator=CalibrationFlowIntegratorConfig(num_steps=FM_INTEGRATION_STEPS, method=FM_INTEGRATION_METHOD, ablation_steps=ABLATION_STEPS),
    loss=CalibrationFlowLossConfig(rotation_scale_rad=np.deg2rad(5.0), translation_scale_m=0.05, time_offset_scale_s=0.05, rotation_weight=2.0, translation_weight=1.0, time_offset_weight=1.0, active_rotation_weight=0.75),
    freeze_backbone_initially=False,
)
fm_model = PooledCalibrationFlowModel(flow_config.backbone, **flow_config.model_kwargs()).to(device=DEVICE, dtype=DTYPE)
load_training_checkpoint(SOURCE_FM_CHECKPOINT, model=fm_model, map_location=DEVICE, restore_rng_state=False)
fm_model.freeze_feature_extractor(False)
fm_loss_weights = flow_config.loss.to_loss_weights()

for module in (fm_model.transformer_encoder, fm_model.pooler, fm_model.mlp_body, fm_model.head):
    assert all(p.requires_grad for p in module.parameters())
print("FM checkpoint loaded:", SOURCE_FM_CHECKPOINT)
print("FM parameters:", sum(p.numel() for p in fm_model.parameters()), "trainable:", sum(p.numel() for p in fm_model.parameters() if p.requires_grad))


In [ ]:
class _CapturedTokens(Exception):
    pass

class _TokenCaptureModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = None
        self.context = None

    def forward(self, tokens, calibration_context=None, **kwargs):
        self.tokens = tokens
        self.context = calibration_context
        raise _CapturedTokens()

geometry_processor = GeometryProcessor()
calibration_updater = CalibrationUpdater()
capture_model = _TokenCaptureModel()

def tokenize_window(window: WindowBatch):
    # Topology-dependent calibration key set/order MUST be decided after augmentation.
    calibration_keys = tuple(window.current_calibration)
    step = WindowStep(geometry_processor=geometry_processor, tokenizer=Tokenizer(measurement_dim=MEASUREMENT_DIM, calibration_keys=calibration_keys), model=capture_model, calibration_updater=calibration_updater)
    capture_model.tokens = None
    capture_model.context = None
    try:
        step(window)
    except _CapturedTokens:
        pass
    if capture_model.tokens is None:
        raise RuntimeError("WindowStep did not call model: failed to capture tokens")
    if tuple(capture_model.tokens.calibration_keys) != calibration_keys:
        raise AssertionError((calibration_keys, capture_model.tokens.calibration_keys))
    return capture_model.tokens, capture_model.context

def flow_supervision(window: WindowBatch, *, generator: torch.Generator, model=None):
    model = fm_model if model is None else model
    with torch.no_grad():
        tokens, context = tokenize_window(window)
        target_xi, target_tau = correction_targets(window)
    output = model.forward_flow_matching(tokens, target_xi, target_tau, calibration_context=context, generator=generator)
    loss = calibration_velocity_loss(output.predicted_velocity, output.flow_batch.target_velocity, weights=fm_loss_weights)
    return loss, output

@torch.no_grad()
def model_predictions(model, window, steps=FM_INTEGRATION_STEPS):
    tokens, context = tokenize_window(window)
    if isinstance(model, PooledCalibrationFlowModel):
        return model(tokens, context, num_steps=steps, method=FM_INTEGRATION_METHOD)
    return model(tokens, context)

from dataclasses import dataclass
from obscalib.flow_matching.lie_integrator import integrate_calibration_flow, CalibrationFlowResult

@dataclass
class JointFMEndpointLoss:
    total: torch.Tensor
    velocity: object
    endpoint: torch.Tensor
    endpoint_rotation: torch.Tensor
    endpoint_translation: torch.Tensor
    endpoint_tau: torch.Tensor
    lidar_active_rotation_deg: torch.Tensor
    lidar_healthy_rotation_deg: torch.Tensor

    def detached_metrics(self):
        metrics = self.velocity.detached_metrics()
        metrics.update({
            "loss_combined_total": float(self.total.detach()),
            "loss_endpoint_total": float(self.endpoint.detach()),
            "loss_endpoint_rotation": float(self.endpoint_rotation.detach()),
            "loss_endpoint_translation": float(self.endpoint_translation.detach()),
            "loss_endpoint_time_offset": float(self.endpoint_tau.detach()),
            "endpoint_lidar_active_rotation_deg": float(self.lidar_active_rotation_deg.detach()),
            "endpoint_lidar_healthy_rotation_deg": float(self.lidar_healthy_rotation_deg.detach()),
        })
        return metrics


def endpoint_loss_from_integration(window, flow_result, keys, target_xi, *, weights):
    """Physical corrected extrinsics and clocks; no detach anywhere on predicted path."""
    current = torch.stack([window.current_calibration[key].transform for key in keys], dim=1)
    desired = torch.stack([window.targets[key].next_transform for key in keys], dim=1)
    current_tau = torch.stack([window.current_calibration[key].time_offset for key in keys], dim=1)
    desired_tau = torch.stack([window.targets[key].next_time_offset for key in keys], dim=1)
    corrected = flow_result.correction_transform @ current
    predicted_tau = current_tau + flow_result.delta_tau

    # SO(3) geodesic angle is stable near identity and bounded by pi.
    residual_rotation = desired[..., :3, :3] @ corrected[..., :3, :3].transpose(-1, -2)
    skew = torch.stack((
        residual_rotation[..., 2, 1] - residual_rotation[..., 1, 2],
        residual_rotation[..., 0, 2] - residual_rotation[..., 2, 0],
        residual_rotation[..., 1, 0] - residual_rotation[..., 0, 1],
    ), dim=-1)
    sin_angle = 0.5 * torch.linalg.vector_norm(skew, dim=-1)
    cos_angle = 0.5 * (residual_rotation.diagonal(dim1=-2, dim2=-1).sum(dim=-1) - 1)
    rotation_angle = torch.atan2(sin_angle, cos_angle)
    translation_error = corrected[..., :3, 3] - desired[..., :3, 3]
    tau_error = predicted_tau - desired_tau

    rotation_loss = (rotation_angle / weights.rotation_scale_rad).square().mean()
    translation_loss = (translation_error / weights.translation_scale_m).square().mean()
    tau_loss = (tau_error / weights.time_offset_scale_s).square().mean()
    combined = weights.rotation_weight * rotation_loss + weights.translation_weight * translation_loss + weights.time_offset_weight * tau_loss

    active_angles, healthy_angles = [], []
    for index, key in enumerate(keys):
        if key.startswith("lidar"):
            active_mask = torch.linalg.vector_norm(target_xi[key][:, :3], dim=-1) > 1e-6
            if bool(active_mask.any()): active_angles.append(torch.rad2deg(rotation_angle[:, index][active_mask]))
            if bool((~active_mask).any()): healthy_angles.append(torch.rad2deg(rotation_angle[:, index][~active_mask]))
    nan_metric = rotation_angle.detach().new_tensor(float("nan"))
    active = torch.cat(active_angles).mean() if active_angles else nan_metric
    healthy = torch.cat(healthy_angles).mean() if healthy_angles else nan_metric
    return combined, rotation_loss, translation_loss, tau_loss, active, healthy


def stratified_flow_grid(reference, num_points, max_time, *, generator=None, randomized=True):
    """Return time queries in [0, max_time], always supervising t=0.

    Evaluation uses a fixed grid; training samples one time in every interior
    stratum rather than learning only a handful of fixed interpolation times.
    """
    if num_points < 2:
        raise ValueError("At least two time points required")
    if randomized:
        increments = (torch.arange(1, num_points, device=reference.device, dtype=reference.dtype) +
                      torch.rand(num_points - 1, device=reference.device, dtype=reference.dtype, generator=generator))
        return torch.cat((reference.new_zeros(1), max_time * increments / num_points))
    return torch.linspace(0, max_time, num_points, device=reference.device, dtype=reference.dtype)


def integrate_training_correction(model, shared, stacked_context, *, generator=None, randomized=True):
    """Train with right-composed Lie-Euler and optionally jittered step boundaries.

    Uses the same update rule as integrate_calibration_flow, with actual dt_k.
    Jitter is disabled in deterministic validation; inference remains unchanged.
    """
    if not randomized:
        return model.integrate_from_features(shared, stacked_context, num_steps=TRAIN_INTEGRATION_STEPS, method=TRAIN_INTEGRATION_METHOD)
    if TRAIN_INTEGRATION_METHOD != "euler":
        raise ValueError("Randomized training intervals currently require Lie-Euler")
    num_steps = TRAIN_INTEGRATION_STEPS
    b, c, _ = shared.shape
    grid = torch.linspace(0, 1, num_steps + 1, device=shared.device, dtype=shared.dtype)
    if num_steps > 1:
        perturb = (torch.rand(num_steps - 1, device=shared.device, dtype=shared.dtype, generator=generator) - 0.5) / num_steps
        grid = torch.cat((grid[:1], grid[1:-1] + perturb, grid[-1:]))
    D = torch.eye(4, dtype=shared.dtype, device=shared.device).expand(b, c, 4, 4).clone()
    tau = shared.new_zeros(b, c, 1)
    for k in range(num_steps):
        dt = grid[k + 1] - grid[k]
        xi = shared.new_zeros(b, c, 6) if k == 0 else se3_log(D.reshape(-1, 4, 4)).reshape(b, c, 6)
        t = grid[k].expand(b, c, 1)
        v = model.velocity_from_features(shared, xi, tau, t, stacked_context)
        D = D @ se3_exp((dt * v[..., :6]).reshape(-1, 6)).reshape(b, c, 4, 4)
        tau = tau + dt * v[..., 6:]
    return CalibrationFlowResult(se3_log(D.reshape(-1, 4, 4)).reshape(b, c, 6), tau, D)


def joint_flow_endpoint_supervision(window, *, generator=None, model=None, objective=None):
    """Correction-to-go velocity MSE and optional differentiable endpoint loss.

    Encode each physical window ONCE and share features over all correction
    queries and integration steps. Targets have no gradients; the predicted
    field retains gradients through all model components.
    """
    model = fm_model if model is None else model
    objective = FULL_OBJECTIVE if objective is None else objective
    with torch.no_grad():
        tokens, context = tokenize_window(window)
        target_xi, target_tau = correction_targets(window)
    shared, stacked_context = model.encode_calibration_features(tokens, context)
    keys = tuple(tokens.calibration_keys)
    delta_xi = torch.stack([target_xi[key] for key in keys], dim=1)
    delta_tau = torch.stack([target_tau[key] for key in keys], dim=1)
    batch_size, num_calibrations, feature_dim = shared.shape
    num_points = FLOW_PATH_NUM_POINTS
    grid = stratified_flow_grid(shared, num_points, objective["max_time"], generator=generator, randomized=model.training and objective["randomized_query_times"])
    times = grid.view(num_points, 1, 1, 1).expand(num_points, batch_size, num_calibrations, 1)
    with torch.no_grad():
        path_batch = make_correction_to_go_flow_batch(
            delta_xi.unsqueeze(0).expand(num_points, -1, -1, -1),
            delta_tau.unsqueeze(0).expand(num_points, -1, -1, -1),
            time=times, mode=objective["mode"],
            off_path_probability=objective["off_path_probability"],
            identity_probability=objective["identity_probability"],
            rotation_noise_std_rad=objective["rotation_noise_std_rad"],
            translation_noise_std_m=objective["translation_noise_std_m"],
            tau_noise_std_s=objective["tau_noise_std_s"],
            max_time=objective["max_time"], generator=generator,
        )
    repeated_shared = shared.unsqueeze(0).expand(num_points, -1, -1, -1).reshape(num_points * batch_size, num_calibrations, feature_dim)
    repeated_context = None if stacked_context is None else stacked_context.unsqueeze(0).expand(num_points, -1, -1, -1).reshape(num_points * batch_size, num_calibrations, stacked_context.shape[-1])
    predicted = model.velocity_from_features(
        repeated_shared,
        path_batch.correction_xi.reshape(num_points * batch_size, num_calibrations, 6),
        path_batch.correction_tau.reshape(num_points * batch_size, num_calibrations, 1),
        path_batch.time.reshape(num_points * batch_size, num_calibrations, 1),
        repeated_context,
    ).reshape(num_points, batch_size, num_calibrations, 7)
    # Active rotation is defined by the original corrupted sensor, NOT by the
    # off-path recovery velocity of an otherwise healthy calibration.
    active_mask = (torch.linalg.vector_norm(delta_xi[..., :3], dim=-1) > fm_loss_weights.active_rotation_threshold_rad)
    active_mask = active_mask.unsqueeze(0).expand(num_points, -1, -1)
    velocity = calibration_velocity_loss(predicted, path_batch.target_velocity, weights=fm_loss_weights, active_rotation_mask=active_mask)

    # The field integrates along its own trajectory, not the teacher path.
    # No torch.no_grad here: train-time endpoint loss must reach the head and backbone.
    result = integrate_training_correction(model, shared, stacked_context, generator=generator,
        randomized=model.training and objective["randomized_integration"])
    endpoint, endpoint_rotation, endpoint_translation, endpoint_tau, active, healthy = endpoint_loss_from_integration(window, result, keys, target_xi, weights=fm_loss_weights)
    total = objective["velocity_weight"] * velocity.total + objective["endpoint_weight"] * endpoint
    if not bool(torch.isfinite(total.detach())):
        raise FloatingPointError("Nonfinite correction-to-go joint loss")
    return JointFMEndpointLoss(total, velocity, endpoint, endpoint_rotation, endpoint_translation, endpoint_tau, active, healthy), result


from obscalib.flow_matching.loss import CalibrationFlowLoss


@dataclass
class OnPolicyJointFMEndpointLoss(JointFMEndpointLoss):
    analytical_velocity_total: torch.Tensor
    on_policy_velocity_total: torch.Tensor

    def detached_metrics(self):
        metrics = super().detached_metrics()
        metrics["loss_fm_analytical_total"] = float(self.analytical_velocity_total.detach())
        metrics["loss_fm_on_policy_total"] = float(self.on_policy_velocity_total.detach())
        return metrics


def _velocity_at_flow_states(model, shared, stacked_context, correction_xi, correction_tau, times):
    """Evaluate the trainable velocity head over multiple correction states."""
    num_points, batch_size, num_calibrations, _ = correction_xi.shape
    hidden_dim = shared.shape[-1]

    features = shared.unsqueeze(0).expand(num_points, -1, -1, -1).reshape(num_points * batch_size, num_calibrations, hidden_dim)
    context = None if stacked_context is None else stacked_context.unsqueeze(0).expand(num_points, -1, -1, -1).reshape(num_points * batch_size, num_calibrations, stacked_context.shape[-1])

    predicted = model.velocity_from_features(
        features,
        correction_xi.reshape(num_points * batch_size, num_calibrations, 6),
        correction_tau.reshape(num_points * batch_size, num_calibrations, 1),
        times.reshape(num_points * batch_size, num_calibrations, 1),
        context,
    )
    return predicted.reshape(num_points, batch_size, num_calibrations, 7)


def _rollout_correction_to_go_states(model, shared, stacked_context, delta_xi, delta_tau, *, num_steps, max_time, method):
    """Integrate the current model without autograd and supervise its visited states."""
    if not (0.0 <= max_time < 1.0):
        raise ValueError("on_policy_max_time must be in [0, 1).")
    if num_steps < 1:
        raise ValueError("on_policy_steps must be positive.")

    with torch.no_grad():
        context = None if stacked_context is None else stacked_context.detach()
        result = model.integrate_from_features(
            shared.detach(), context,
            num_steps=num_steps, method=method, record_trajectory=True,
        )

        states = tuple(state for state in result.trajectory if state.time <= max_time + 1e-7)
        if not states or abs(states[0].time) > 1e-7:
            raise RuntimeError("Rollout trajectory must begin at identity.")

        num_points = len(states)
        batch_size, num_calibrations, _ = delta_xi.shape

        D = torch.stack([state.correction_transform for state in states], dim=0)
        tau = torch.stack([state.correction_tau for state in states], dim=0)
        times = delta_tau.new_tensor([state.time for state in states]).view(num_points, 1, 1, 1).expand(num_points, batch_size, num_calibrations, 1)

        target_D = se3_exp(delta_xi.reshape(-1, 6)).reshape(batch_size, num_calibrations, 4, 4)
        remaining_D = torch.linalg.solve(D, target_D.unsqueeze(0).expand_as(D))
        remaining_xi = se3_log(remaining_D.reshape(-1, 4, 4)).reshape(num_points, batch_size, num_calibrations, 6)

        remaining_time = 1.0 - times
        target_velocity = torch.cat(
            (remaining_xi / remaining_time, (delta_tau.unsqueeze(0) - tau) / remaining_time),
            dim=-1,
        )
        correction_xi = se3_log(D.reshape(-1, 4, 4)).reshape(num_points, batch_size, num_calibrations, 6)

        if not bool(torch.isfinite(target_velocity).all()):
            raise FloatingPointError("Nonfinite on-policy correction-to-go target.")

        return correction_xi, tau, times, target_velocity


def _mix_calibration_velocity_losses(analytical, on_policy, fraction):
    """Retain the existing CalibrationFlowLoss interface and metrics."""
    a, b = 1.0 - fraction, fraction
    return CalibrationFlowLoss(
        total=a * analytical.total + b * on_policy.total,
        rotation=a * analytical.rotation + b * on_policy.rotation,
        translation=a * analytical.translation + b * on_policy.translation,
        time_offset=a * analytical.time_offset + b * on_policy.time_offset,
        active_rotation=a * analytical.active_rotation + b * on_policy.active_rotation,
        num_active_rotations=analytical.num_active_rotations + on_policy.num_active_rotations,
    )


def joint_flow_endpoint_supervision(window, *, generator=None, model=None, objective=None):
    """Analytical CTG + detached on-policy CTG + differentiable endpoint loss."""
    model = fm_model if model is None else model
    objective = FULL_OBJECTIVE if objective is None else objective
    fraction = float(objective.get("on_policy_fraction", 0.5))

    if not (0.0 <= fraction <= 1.0):
        raise ValueError("on_policy_fraction must be in [0, 1].")

    with torch.no_grad():
        tokens, context = tokenize_window(window)
        target_xi, target_tau = correction_targets(window)

    # Compute sensor features ONCE. Gradients through these features are retained.
    shared, stacked_context = model.encode_calibration_features(tokens, context)
    keys = tuple(tokens.calibration_keys)
    delta_xi = torch.stack([target_xi[key] for key in keys], dim=1)
    delta_tau = torch.stack([target_tau[key] for key in keys], dim=1)

    batch_size, num_calibrations, _ = shared.shape
    active = torch.linalg.vector_norm(delta_xi[..., :3], dim=-1) > fm_loss_weights.active_rotation_threshold_rad

    # ---------------------------------------------------------
    # 1. Original analytical correction-to-go supervision
    # ---------------------------------------------------------

    analytical_loss = None

    if fraction < 1.0:
        num_points = FLOW_PATH_NUM_POINTS
        grid = stratified_flow_grid(
            shared, num_points, objective["max_time"], generator=generator,
            randomized=model.training and objective["randomized_query_times"],
        )
        times = grid.view(num_points, 1, 1, 1).expand(num_points, batch_size, num_calibrations, 1)

        with torch.no_grad():
            path = make_correction_to_go_flow_batch(
                delta_xi.unsqueeze(0).expand(num_points, -1, -1, -1),
                delta_tau.unsqueeze(0).expand(num_points, -1, -1, -1),
                time=times,
                mode=objective["mode"],
                off_path_probability=objective["off_path_probability"],
                identity_probability=objective["identity_probability"],
                rotation_noise_std_rad=objective["rotation_noise_std_rad"],
                translation_noise_std_m=objective["translation_noise_std_m"],
                tau_noise_std_s=objective["tau_noise_std_s"],
                max_time=objective["max_time"],
                generator=generator,
            )

        predicted = _velocity_at_flow_states(
            model, shared, stacked_context,
            path.correction_xi, path.correction_tau, path.time,
        )
        analytical_loss = calibration_velocity_loss(
            predicted, path.target_velocity,
            weights=fm_loss_weights,
            active_rotation_mask=active.unsqueeze(0).expand(num_points, -1, -1),
        )

    # ---------------------------------------------------------
    # 2. Generate detached states from model's OWN trajectory
    # ---------------------------------------------------------

    on_policy_loss = None

    if fraction > 0.0:
        rollout_xi, rollout_tau, rollout_times, rollout_target = _rollout_correction_to_go_states(
            model, shared, stacked_context, delta_xi, delta_tau,
            num_steps=int(objective.get("on_policy_steps", 8)),
            max_time=float(objective.get("on_policy_max_time", 0.875)),
            method=objective.get("on_policy_method", "midpoint"),
        )

        # This forward pass DOES retain gradients through all neural components.
        rollout_predicted = _velocity_at_flow_states(
            model, shared, stacked_context,
            rollout_xi, rollout_tau, rollout_times,
        )

        on_policy_loss = calibration_velocity_loss(
            rollout_predicted, rollout_target,
            weights=fm_loss_weights,
            active_rotation_mask=active.unsqueeze(0).expand(rollout_xi.shape[0], -1, -1),
        )

    # ---------------------------------------------------------
    # 3. Combine both velocity objectives
    # ---------------------------------------------------------

    if analytical_loss is None:
        velocity = on_policy_loss
    elif on_policy_loss is None:
        velocity = analytical_loss
    else:
        velocity = _mix_calibration_velocity_losses(analytical_loss, on_policy_loss, fraction)

    # ---------------------------------------------------------
    # 4. Original differentiable 4-step endpoint objective
    # ---------------------------------------------------------

    result = integrate_training_correction(
        model, shared, stacked_context,
        generator=generator,
        randomized=model.training and objective["randomized_integration"],
    )

    endpoint, endpoint_rotation, endpoint_translation, endpoint_tau, lidar_active, lidar_healthy = endpoint_loss_from_integration(
        window, result, keys, target_xi, weights=fm_loss_weights,
    )

    total = objective["velocity_weight"] * velocity.total + objective["endpoint_weight"] * endpoint
    if not bool(torch.isfinite(total.detach())):
        raise FloatingPointError("Nonfinite on-policy joint loss.")

    zero = total.detach().new_zeros(())
    losses = OnPolicyJointFMEndpointLoss(
        total=total,
        velocity=velocity,
        endpoint=endpoint,
        endpoint_rotation=endpoint_rotation,
        endpoint_translation=endpoint_translation,
        endpoint_tau=endpoint_tau,
        lidar_active_rotation_deg=lidar_active,
        lidar_healthy_rotation_deg=lidar_healthy,
        analytical_velocity_total=analytical_loss.total if analytical_loss is not None else zero,
        on_policy_velocity_total=on_policy_loss.total if on_policy_loss is not None else zero,
    )
    return losses, result


In [ ]:

def remaining_displacement_velocity_loss(predicted, target, times, active_mask):
    """Time-weighted CTG loss, with branch-wise weight normalization.

    The target remains velocity; multiplying BOTH arguments by remaining time
    penalizes predicted remaining displacement without changing the pointwise optimum.
    """
    if predicted.shape != target.shape or times.shape != predicted.shape[:-1] + (1,):
        raise ValueError("Inconsistent predicted, target, or time shapes")

    remaining = 1.0 - times
    if bool((remaining <= 0).any()):
        raise ValueError("Correction-to-go supervision requires t < 1")

    # Keep the mean squared time weight equal to 1 for each branch.
    scale = remaining / remaining.square().mean().sqrt().clamp_min(1e-8)

    return calibration_velocity_loss(
        predicted * scale,
        target * scale,
        weights=fm_loss_weights,
        active_rotation_mask=active_mask,
    )


def joint_flow_endpoint_supervision(window, *, generator=None, model=None, objective=None):
    """Normalized remaining-displacement supervision on analytical and rollout states.

    Sensor features are encoded once. On-policy trajectory generation is detached,
    but supervised velocity predictions retain gradients through all model blocks.
    """
    model = fm_model if model is None else model
    objective = FULL_OBJECTIVE if objective is None else objective
    fraction = float(objective.get("on_policy_fraction", 0.5))

    if not (0.0 <= fraction <= 1.0):
        raise ValueError("on_policy_fraction must be in [0, 1].")

    with torch.no_grad():
        tokens, context = tokenize_window(window)
        target_xi, target_tau = correction_targets(window)

    shared, stacked_context = model.encode_calibration_features(tokens, context)
    keys = tuple(tokens.calibration_keys)
    delta_xi = torch.stack([target_xi[key] for key in keys], dim=1)
    delta_tau = torch.stack([target_tau[key] for key in keys], dim=1)

    batch_size, num_calibrations, _ = shared.shape
    active = torch.linalg.vector_norm(delta_xi[..., :3], dim=-1) > fm_loss_weights.active_rotation_threshold_rad

    # ---------------------------------------------------------
    # 1. Analytical CTG supervision, up to t=0.975
    # ---------------------------------------------------------

    analytical_loss = None

    if fraction < 1.0:
        num_points = FLOW_PATH_NUM_POINTS
        grid = stratified_flow_grid(
            shared, num_points, objective["max_time"],
            generator=generator,
            randomized=model.training and objective["randomized_query_times"],
        )
        times = grid.view(num_points, 1, 1, 1).expand(num_points, batch_size, num_calibrations, 1)

        with torch.no_grad():
            path = make_correction_to_go_flow_batch(
                delta_xi.unsqueeze(0).expand(num_points, -1, -1, -1),
                delta_tau.unsqueeze(0).expand(num_points, -1, -1, -1),
                time=times,
                mode=objective["mode"],
                off_path_probability=objective["off_path_probability"],
                identity_probability=objective["identity_probability"],
                rotation_noise_std_rad=objective["rotation_noise_std_rad"],
                translation_noise_std_m=objective["translation_noise_std_m"],
                tau_noise_std_s=objective["tau_noise_std_s"],
                max_time=objective["max_time"],
                generator=generator,
            )

        predicted = _velocity_at_flow_states(
            model, shared, stacked_context,
            path.correction_xi, path.correction_tau, path.time,
        )

        analytical_loss = remaining_displacement_velocity_loss(
            predicted,
            path.target_velocity,
            path.time,
            active.unsqueeze(0).expand(num_points, -1, -1),
        )

    # ---------------------------------------------------------
    # 2. On-policy CTG supervision, up to t=0.875
    # ---------------------------------------------------------

    on_policy_loss = None

    if fraction > 0.0:
        rollout_xi, rollout_tau, rollout_times, rollout_target = _rollout_correction_to_go_states(
            model, shared, stacked_context, delta_xi, delta_tau,
            num_steps=int(objective.get("on_policy_steps", 8)),
            max_time=float(objective.get("on_policy_max_time", 0.875)),
            method=objective.get("on_policy_method", "midpoint"),
        )

        rollout_predicted = _velocity_at_flow_states(
            model, shared, stacked_context,
            rollout_xi, rollout_tau, rollout_times,
        )

        on_policy_loss = remaining_displacement_velocity_loss(
            rollout_predicted,
            rollout_target,
            rollout_times,
            active.unsqueeze(0).expand(rollout_xi.shape[0], -1, -1),
        )

    # ---------------------------------------------------------
    # 3. Combine remaining-displacement losses
    # ---------------------------------------------------------

    if analytical_loss is None:
        velocity = on_policy_loss
    elif on_policy_loss is None:
        velocity = analytical_loss
    else:
        velocity = _mix_calibration_velocity_losses(analytical_loss, on_policy_loss, fraction)

    # ---------------------------------------------------------
    # 4. Differentiable endpoint loss, unchanged
    # ---------------------------------------------------------

    result = integrate_training_correction(
        model, shared, stacked_context,
        generator=generator,
        randomized=model.training and objective["randomized_integration"],
    )

    endpoint, endpoint_rotation, endpoint_translation, endpoint_tau, lidar_active, lidar_healthy = endpoint_loss_from_integration(
        window, result, keys, target_xi, weights=fm_loss_weights,
    )

    total = objective["velocity_weight"] * velocity.total + objective["endpoint_weight"] * endpoint

    if not bool(torch.isfinite(total.detach())):
        raise FloatingPointError("Nonfinite correction-to-go joint loss")

    zero = total.detach().new_zeros(())
    losses = OnPolicyJointFMEndpointLoss(
        total=total,
        velocity=velocity,
        endpoint=endpoint,
        endpoint_rotation=endpoint_rotation,
        endpoint_translation=endpoint_translation,
        endpoint_tau=endpoint_tau,
        lidar_active_rotation_deg=lidar_active,
        lidar_healthy_rotation_deg=lidar_healthy,
        analytical_velocity_total=analytical_loss.total if analytical_loss is not None else zero,
        on_policy_velocity_total=on_policy_loss.total if on_policy_loss is not None else zero,
    )

    return losses, result


## 6. Source-preserving geometry, oracle, and gradient checks

In [ ]:
test_base = window_batch_to(next(iter(train_loader)), DEVICE, dtype=DTYPE)
with torch.no_grad():
    base = augment_original_window(test_base, augmentation_generator(TRAIN_AUG_SEED+1), regime="fine")
    rotated, H = rotate_yaw_view(base, augmentation_generator(TRAIN_AUG_SEED+2), make_yaw_randomizer(180., 1.))
    R = H[:, :3, :3]
    for stream_key in base.streams:
        a, b = base.streams[stream_key], rotated.streams[stream_key]
        torch.testing.assert_close(a.values, b.values, rtol=0, atol=0)
        torch.testing.assert_close(a.timestamps, b.timestamps, rtol=0, atol=0)
        if a.interval_start_timestamps is not None:
            torch.testing.assert_close(a.interval_start_timestamps, b.interval_start_timestamps, rtol=0, atol=0)
    for key in CALIBRATION_KEYS:
        ac, bc = base.current_calibration[key], rotated.current_calibration[key]
        torch.testing.assert_close(bc.transform, H @ ac.transform, rtol=3e-5, atol=3e-5)
        torch.testing.assert_close(bc.time_offset, ac.time_offset, rtol=0, atol=0)
        ta, tb = base.targets[key], rotated.targets[key]
        torch.testing.assert_close(tb.next_transform, H @ ta.next_transform, rtol=3e-5, atol=3e-5)
        xi_a, tau_a = correction_targets(base)
        xi_b, tau_b = correction_targets(rotated)
        torch.testing.assert_close(xi_b[key], rotate_twists(xi_a[key], H), rtol=3e-3, atol=4e-4)
        torch.testing.assert_close(tau_b[key], tau_a[key], rtol=0, atol=1e-6)
        for other in CALIBRATION_KEYS:
            ra = torch.linalg.solve(ac.transform, base.current_calibration[other].transform)
            rb = torch.linalg.solve(bc.transform, rotated.current_calibration[other].transform)
            torch.testing.assert_close(ra, rb, atol=5e-5, rtol=5e-5)
    for window in (base, rotated):
        xi, tau = correction_targets(window)
        tokens, context = tokenize_window(window)
        assert tuple(tokens.calibration_keys) == CALIBRATION_KEYS and tokens.x.shape[-1] == 8
        target_vel = torch.stack([torch.cat((xi[key], tau[key]), dim=-1) for key in CALIBRATION_KEYS], dim=1)
        ref = target_vel.new_zeros(*target_vel.shape[:-1], 8)
        oracle = integrate_calibration_flow(lambda state_xi, state_tau, t: target_vel, ref, num_steps=4, method="euler")
        for j, key in enumerate(CALIBRATION_KEYS):
            torch.testing.assert_close(oracle.correction_transform[:, j] @ window.current_calibration[key].transform, window.targets[key].next_transform, atol=5e-4, rtol=5e-4)

fm_model.train()
fm_model.zero_grad(set_to_none=True)
grad_loss, _ = joint_flow_endpoint_supervision(rotated, generator=augmentation_generator(TRAIN_AUG_SEED+3))
grad_loss.total.backward()
for component in ("transformer_encoder", "pooler", "mlp_body", "head"):
    grads = [p.grad for p in getattr(fm_model, component).parameters() if p.requires_grad and p.grad is not None]
    assert grads and all(torch.isfinite(g).all() for g in grads)
    print(component, "gradient norm", f"{sum(g.square().sum().item() for g in grads)**0.5:.4g}")
fm_model.zero_grad(set_to_none=True)
fm_model.eval()
print("PASS: raw measurements unchanged, physical SE3 and tau truth consistently rotated, oracle endpoints, gradients")


### 6a. Corruption sanity and scale diagnostic (actual KAIST edges)

In [ ]:
# Preflight: corruption MUST touch only LiDAR values, and paired yaw views MUST
# share precisely the same corrupted raw stream. Show actual scale and edge noise.

with torch.no_grad():
    sanity_generator = augmentation_generator(TRAIN_AUG_SEED + 404)
    sanity_base = window_batch_to(next(iter(train_loader)), DEVICE, dtype=DTYPE)
    sanity_physical = augment_original_window(sanity_base, sanity_generator, regime="fine")
    sanity_noise, sanity_info = corrupt_lidar_window(sanity_physical, augmentation_generator(TRAIN_AUG_SEED + 405), augmenter=odometry_noise_augmenter(1.0))
    sanity_rotated, _ = rotate_window_fixed_yaw(sanity_noise, 180.0)
    assert set(sanity_noise.current_calibration) == set(sanity_physical.current_calibration)
    summary_rows = []
    for key, raw in sanity_physical.streams.items():
        noisy = sanity_noise.streams[key]
        assert torch.equal(raw.sample_mask, noisy.sample_mask)
        torch.testing.assert_close(raw.timestamps, noisy.timestamps, atol=0, rtol=0)
        if raw.interval_start_timestamps is not None:
            torch.testing.assert_close(raw.interval_start_timestamps, noisy.interval_start_timestamps, atol=0, rtol=0)
        torch.testing.assert_close(noisy.values, sanity_rotated.streams[key].values, atol=0, rtol=0)
        invalid = ~raw.sample_mask
        if bool(invalid.any()): torch.testing.assert_close(raw.values[invalid], noisy.values[invalid], atol=0, rtol=0)
        if key not in LIDAR_STREAM_KEYS:
            torch.testing.assert_close(raw.values, noisy.values, atol=0, rtol=0)
            continue
        valid = raw.sample_mask
        rel = noisy.values[..., :3, :3] @ raw.values[..., :3, :3].transpose(-1, -2)
        skew = torch.stack((rel[..., 2, 1]-rel[..., 1, 2], rel[..., 0, 2]-rel[..., 2, 0], rel[..., 1, 0]-rel[..., 0, 1]), -1)
        err_rad = torch.atan2(.5*torch.linalg.vector_norm(skew, dim=-1), .5*(rel.diagonal(dim1=-2,dim2=-1).sum(dim=-1)-1))
        edge_err = torch.rad2deg(err_rad[valid])
        scale = sanity_info[key]["translation_scale"]
        assert torch.isfinite(noisy.values[valid]).all()
        # Compare accumulated rotation over up to 20 consecutive valid LiDAR edges.
        chain_errors = []
        for b in range(valid.shape[0]):
            valid_edges = valid[b].nonzero(as_tuple=True)[0]
            for start in range(0, max(0, len(valid_edges)-19), 20):
                edges = valid_edges[start:start+20]
                if len(edges) != 20 or not bool(torch.all(edges[1:] == edges[:-1]+1)): continue
                A = torch.eye(3, device=DEVICE, dtype=DTYPE)
                B = torch.eye(3, device=DEVICE, dtype=DTYPE)
                for k in edges:
                    A = A @ raw.values[b,k,:3,:3]
                    B = B @ noisy.values[b,k,:3,:3]
                Rchain = B @ A.transpose(-1,-2)
                chain_sine = .5*torch.linalg.vector_norm(torch.stack((Rchain[2,1]-Rchain[1,2],Rchain[0,2]-Rchain[2,0],Rchain[1,0]-Rchain[0,1])))
                chain_cosine = .5*(torch.trace(Rchain)-1)
                chain_errors.append(float(torch.rad2deg(torch.atan2(chain_sine, chain_cosine))))
        summary_rows.append(dict(stream=key, n_edges=int(valid.sum()), median_corruption_deg=float(edge_err.median()),
                                 p90_corruption_deg=float(torch.quantile(edge_err, .9)),
                                 median_20edge_drift_deg=float(np.median(chain_errors)) if chain_errors else float("nan"),
                                 n_20edge_chains=len(chain_errors),
                                 scale_mean=float(scale.mean()), scale_min=float(scale.min()), scale_max=float(scale.max())))
    # Strict identity check: zero noise/scale and zero intensity must not alter measurements.
    untouched, _ = corrupt_lidar_window(sanity_physical, augmentation_generator(TRAIN_AUG_SEED+406), augmenter=odometry_noise_augmenter(0.0))
    for key in sanity_physical.streams:
        torch.testing.assert_close(untouched.streams[key].values, sanity_physical.streams[key].values, atol=0, rtol=0)
    for key in CALIBRATION_KEYS:
        torch.testing.assert_close(sanity_noise.current_calibration[key].transform, sanity_physical.current_calibration[key].transform, atol=0, rtol=0)
        torch.testing.assert_close(sanity_noise.targets[key].next_transform, sanity_physical.targets[key].next_transform, atol=0, rtol=0)
    display(pd.DataFrame(summary_rows).round(4))
    print("Compare measured New College LiDAR/gyro disagreement: median ~0.307 deg/edge; its 5.14 deg/20-edge residual also contains non-ICP terms.")
    print("PASS: LiDAR only, masks/time/targets invariant, identical raw data under common yaw")

del sanity_base, sanity_physical, sanity_noise, sanity_rotated, sanity_info
if DEVICE.type == "cuda": torch.cuda.empty_cache()


## 7. Paired FM objective and correctly normalized fine/coarse weighting

In [ ]:
def set_coarse_angle(angle_deg):
    """Rebuild only the coarse uniform prior; do not change the fine distribution."""
    global COARSE_ROTATION_MAX_RAD, coarse_prior_config, coarse_prior_perturber
    angle_deg = float(angle_deg)
    if not (60.0 <= angle_deg < 180.0):
        raise ValueError("Coarse maximum angle must be in [60,180) degrees")
    COARSE_ROTATION_MAX_RAD = np.deg2rad(angle_deg)
    coarse_prior_config = replace(coarse_prior_config, rotation=replace(coarse_prior_config.rotation, maximum=COARSE_ROTATION_MAX_RAD))
    coarse_prior_perturber = CalibrationPriorPerturber(coarse_prior_config)


def weighted_regime_groups(base_window, generator, regime):
    """Split a homogeneous topology batch before augmentation, recording the actual sampled regime."""
    if regime not in ("mixed", "fine", "coarse"):
        raise ValueError(regime)
    n = len(next(iter(base_window.current_calibration.values())).transform)
    if regime != "mixed":
        return [(regime, base_window)]
    device = next(iter(base_window.current_calibration.values())).transform.device
    fine_mask = torch.rand(n, device=device, generator=generator) < FINE_PRIOR_PROBABILITY
    return [(name, topology_slice_batch(base_window, positions))
            for name, positions in (("fine", fine_mask.nonzero(as_tuple=True)[0]),
                                    ("coarse", (~fine_mask).nonzero(as_tuple=True)[0])) if positions.numel()]


def regime_loss_multiplier(regime, sampling_regime, fine_weight, coarse_weight):
    """Importance-adjusted multiplier for alpha_f * E[L_f] + alpha_c * E[L_c].

    For mixed sampling p(fine)=FINE_PRIOR_PROBABILITY, alpha weights are explicit
    desired objective weights. Setting both to .5 yields the original unbiased
    objective with 50/50 sampling. Single-regime training uses multiplier 1.
    """
    if sampling_regime != "mixed":
        return 1.0
    probability = FINE_PRIOR_PROBABILITY if regime == "fine" else 1.0 - FINE_PRIOR_PROBABILITY
    weight = fine_weight if regime == "fine" else coarse_weight
    return float(weight / (fine_weight + coarse_weight) / probability)

# At fixed 50/50 sampling, these settings recover the original objective.
assert np.isclose(regime_loss_multiplier("fine", "mixed", 0.5, 0.5), 1.0)
assert np.isclose(regime_loss_multiplier("coarse", "mixed", 0.5, 0.5), 1.0)
set_coarse_angle(FULL_COARSE_MAX_DEG)

def paired_endpoint_equivariance(base_result, rotated_result, H):
    """Equivariance of INTEGRATED correction, not just intermediate velocities.

    Correct transformation: D_rot = H @ D_base @ inv(H).  Rotation-only H.
    Angular and translation components use the same normalization as FM loss.
    """
    D0, D1 = base_result.correction_transform, rotated_result.correction_transform
    expected = H.unsqueeze(1) @ D0 @ torch.linalg.inv(H).unsqueeze(1)
    residual = torch.linalg.solve(expected, D1)
    twist = se3_log(residual.reshape(-1, 4, 4)).reshape(*D0.shape[:2], 6)
    rot_err = (twist[..., :3] / fm_loss_weights.rotation_scale_rad).square().mean()
    trans_err = (twist[..., 3:] / fm_loss_weights.translation_scale_m).square().mean()
    tau_err = ((base_result.delta_tau - rotated_result.delta_tau) / fm_loss_weights.time_offset_scale_s).square().mean()
    return fm_loss_weights.rotation_weight * rot_err + fm_loss_weights.translation_weight * trans_err + fm_loss_weights.time_offset_weight * tau_err


def evaluate_paired_objective(window, *, generator, frame_randomizer, regime, objective, paired=True):
    """Two supervised coordinate views of one physical sample; optional equivariance."""
    rotated, H = rotate_yaw_view(window, generator, frame_randomizer)
    loss_rotated, result_rotated = joint_flow_endpoint_supervision(rotated, generator=generator, objective=objective)
    if not paired:
        return loss_rotated.total, dict(rotated=float(loss_rotated.total.detach()), original=float('nan'), equivariance=0.0, endpoint=float(loss_rotated.endpoint.detach()), velocity=float(loss_rotated.velocity.total.detach()), lidar_active=float(loss_rotated.lidar_active_rotation_deg.detach()), lidar_healthy=float(loss_rotated.lidar_healthy_rotation_deg.detach()))
    loss_original, result_original = joint_flow_endpoint_supervision(window, generator=generator, objective=objective)
    equiv = paired_endpoint_equivariance(result_original, result_rotated, H) if EQUIVARIANCE_LOSS_WEIGHT else loss_original.total.new_zeros(())
    supervised = (loss_original.total + ROTATED_VIEW_WEIGHT * loss_rotated.total) / (1. + ROTATED_VIEW_WEIGHT)
    total = supervised + EQUIVARIANCE_LOSS_WEIGHT * equiv
    return total, dict(
        rotated=float(loss_rotated.total.detach()), original=float(loss_original.total.detach()),
        equivariance=float(equiv.detach()), endpoint=float(loss_rotated.endpoint.detach()),
        velocity=float(loss_rotated.velocity.total.detach()),
        lidar_active=float(loss_rotated.lidar_active_rotation_deg.detach()),
        lidar_healthy=float(loss_rotated.lidar_healthy_rotation_deg.detach()),
    )

set_coarse_angle(FULL_COARSE_MAX_DEG)


## 8. Stochastic preflight: fixed clean/noisy validation, optional noise curriculum

In [ ]:
@torch.no_grad()
def fixed_precheck_evaluation(model, eval_windows):
    """Evaluate identical clean/noisy cases under original and fixed yaw frames."""
    was_training = model.training
    model.eval()
    rows = []
    try:
        for (noise_condition, regime, yaw), window in eval_windows.items():
            prediction = model_predictions(model, window, steps=FM_INTEGRATION_STEPS)
            xi, _ = correction_targets(window)
            for key in CALIBRATION_KEYS:
                current = window.current_calibration[key].transform
                desired = window.targets[key].next_transform
                corrected = se3_exp(prediction.predictions[key].delta_xi) @ current
                R = desired[:, :3, :3] @ corrected[:, :3, :3].transpose(-1, -2)
                sine = .5*torch.linalg.vector_norm(torch.stack((R[:,2,1]-R[:,1,2],R[:,0,2]-R[:,2,0],R[:,1,0]-R[:,0,1]),-1),dim=-1)
                cosine = .5*(R.diagonal(dim1=-2,dim2=-1).sum(dim=-1)-1)
                errors = torch.rad2deg(torch.atan2(sine, cosine))
                active = torch.linalg.vector_norm(xi[key][:,:3],dim=-1) > 1e-6
                for role, mask in (("active",active),("healthy",~active)):
                    if bool(mask.any()):
                        rows.append(dict(noise=noise_condition, regime=regime, yaw_deg=yaw, key=key, role=role,
                                         mean_deg=float(errors[mask].mean()), n=int(mask.sum())))
    finally:
        model.train(was_training)
    return pd.DataFrame(rows)


def fixed_precheck_score(metrics):
    """Mean over yaw/regime/LiDAR key groups; balance clean versus corrupt explicitly."""
    focus = metrics[(metrics.role == "active") & (metrics.key.str.startswith("lidar"))]
    by_noise = focus.groupby("noise").mean_deg.mean()
    if set(by_noise.index) != {"clean", "corrupt"}: raise RuntimeError("Both noise validation conditions required")
    return float((1.0 - VAL_NOISY_WEIGHT)*by_noise["clean"] + VAL_NOISY_WEIGHT*by_noise["corrupt"])


STOCHASTIC_CHECK_PASSED = None
precheck_history, precheck_final = [], None

if DEVICE.type == "cuda": torch.cuda.empty_cache()
if RUN_STOCHASTIC_CHECK:
    # Use the same source checkpoint for every stochastic experiment.
    sample_indices = np.unique(np.linspace(0, len(train_dataset)-1, min(STOCHASTIC_WINDOWS, len(train_dataset))).astype(int))
    small_base = window_batch_to(collate_windows([train_dataset[int(i)] for i in sample_indices]), DEVICE, dtype=DTYPE)
    previous_coarse_deg = np.rad2deg(COARSE_ROTATION_MAX_RAD)
    set_coarse_angle(STOCHASTIC_COARSE_MAX_DEG)  # Fixed coarse evaluation uses the stochastic maximum, not FULL_COARSE_MAX_DEG.
    fixed_cases = {}
    # Fixed cases are constructed once, always with FULL noise intensity for validation.
    # Clean/noisy share exactly the same physical window and calibration perturbation.
    for reg_idx, regime in enumerate(("fine", "coarse")):
        physical = augment_original_window(small_base, augmentation_generator(STOCHASTIC_EVAL_SEED+reg_idx), regime=regime)
        noisy, _ = corrupt_lidar_window(physical, augmentation_generator(STOCHASTIC_EVAL_SEED+100+reg_idx), augmenter=odometry_noise_augmenter(1.0))
        for condition, sample in (("clean", physical), ("corrupt", noisy)):
            for yaw in (0.0, 90.0, 180.0, 270.0):
                fixed_cases[(condition,regime,yaw)] = rotate_window_fixed_yaw(sample, yaw)[0]

    saved_weights = {k: v.detach().cpu().clone() for k,v in fm_model.state_dict().items()}
    saved_mode = fm_model.training
    saved_cpu_rng = torch.random.get_rng_state().clone()
    saved_cuda_rng = torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else None
    saved_numpy_rng, saved_python_rng = np.random.get_state(), random.getstate()
    diagnostic_optimizer = None
    try:
        load_training_checkpoint(SOURCE_FM_CHECKPOINT, model=fm_model, map_location=DEVICE, restore_rng_state=False)
        set_coarse_angle(STOCHASTIC_COARSE_MAX_DEG)
        diagnostic_optimizer = torch.optim.AdamW([
            {"params": list(fm_model.head.parameters()), "lr": STOCHASTIC_HEAD_LR},
            {"params": [p for block in (fm_model.transformer_encoder, fm_model.pooler, fm_model.mlp_body) for p in block.parameters()], "lr": STOCHASTIC_BACKBONE_LR},
        ],weight_decay=WEIGHT_DECAY)
        train_generator = augmentation_generator(STOCHASTIC_SEED)
        train_noise_generator = augmentation_generator(STOCHASTIC_SEED+1)
        before = fixed_precheck_evaluation(fm_model, fixed_cases)
        best_candidate, best_score, best_step = None, float("inf"), None
        n_base = len(next(iter(small_base.current_calibration.values())).transform)

        for step in range(1, STOCHASTIC_STEPS+1):
            intensity = noise_intensity(step, schedule=STOCHASTIC_NOISE_SCHEDULE, target=STOCHASTIC_NOISE_INTENSITY,
                                         ramp_start=STOCHASTIC_RAMP_START_INTENSITY, ramp_length=STOCHASTIC_RAMP_UPDATES)
            noise_augmenter = odometry_noise_augmenter(intensity)
            yaw_randomizer = make_yaw_randomizer(STOCHASTIC_YAW_MAX_DEG, STOCHASTIC_YAW_PROBABILITY)
            fm_model.train(); diagnostic_optimizer.zero_grad(set_to_none=True)
            train_total, clean_count, noisy_count = 0.0, 0, 0
            for _ in range(GRADIENT_ACCUMULATION_STEPS):
                with torch.no_grad():
                    groups = weighted_regime_groups(small_base, train_generator, STOCHASTIC_TRAIN_REGIME)
                for regime, subgroup in groups:
                    with torch.no_grad():
                        calibrated = augment_original_window(subgroup, train_generator, regime=regime)
                        noise_parts = noise_subgroups(calibrated, train_noise_generator, ODOM_CORRUPTION_PROBABILITY)
                    for noise_mode, sample in noise_parts:
                        with torch.no_grad():
                            augmented, _ = corrupt_lidar_window(sample, train_noise_generator, noise_mode=noise_mode, augmenter=noise_augmenter)
                        loss, details = evaluate_paired_objective(augmented, generator=train_generator, frame_randomizer=yaw_randomizer,
                                                                 regime=regime, objective=STOCHASTIC_OBJECTIVE, paired=STOCHASTIC_PAIRED_VIEWS)
                        n = len(next(iter(augmented.current_calibration.values())).transform)
                        weight = n*regime_loss_multiplier(regime, STOCHASTIC_TRAIN_REGIME, STOCHASTIC_FINE_LOSS_WEIGHT, STOCHASTIC_COARSE_LOSS_WEIGHT)/(n_base*GRADIENT_ACCUMULATION_STEPS)
                        (weight*loss).backward()
                        train_total += float(loss.detach())*weight
                        clean_count += n if noise_mode == "clean" else 0
                        noisy_count += n if noise_mode == "corrupt" else 0
            grad_before_clip = torch.nn.utils.clip_grad_norm_(fm_model.parameters(), GRADIENT_CLIP_NORM)
            diagnostic_optimizer.step()
            if step == 1 or step % STOCHASTIC_REPORT_EVERY == 0 or step == STOCHASTIC_STEPS:
                metrics = fixed_precheck_evaluation(fm_model,fixed_cases)
                metrics["step"] = step
                metrics["noise_intensity"] = intensity
                metrics["stochastic_train_loss"] = train_total
                metrics["grad_norm_before_clip"] = float(grad_before_clip)
                precheck_history.extend(metrics.to_dict("records"))
                score = fixed_precheck_score(metrics)
                if np.isfinite(score) and score < best_score:
                    best_score, best_step = score, step
                    best_candidate = {k:v.detach().cpu().clone() for k,v in fm_model.state_dict().items()}
                active_lidar = metrics[(metrics.role=="active") & metrics.key.str.startswith("lidar")]
                for_noise = active_lidar.groupby("noise").mean_deg.mean().to_dict()
                print(f"Step {step:3d}/{STOCHASTIC_STEPS} noise={intensity:.2f} train={train_total:.3f} grad={float(grad_before_clip):.2f} "
                      f"| clean={for_noise.get('clean',float('nan')):.2f}° noisy={for_noise.get('corrupt',float('nan')):.2f}° "
                      f"weighted={score:.2f}° | sampled clean/noisy={clean_count}/{noisy_count}", flush=True)
        after = fixed_precheck_evaluation(fm_model, fixed_cases)
        precheck_final = before.merge(after,on=["noise","regime","yaw_deg","key","role"],suffixes=("_before","_after"))
        precheck_final["delta_deg"] = precheck_final.mean_deg_after-precheck_final.mean_deg_before
        display(precheck_final.round(3))
        before_score, after_score = fixed_precheck_score(before), fixed_precheck_score(after)
        STOCHASTIC_CHECK_PASSED = bool(np.isfinite(after_score) and after_score < before_score)
        print("STOCHASTIC:", "PASS" if STOCHASTIC_CHECK_PASSED else "INCONCLUSIVE",
              f"weighted active LiDAR: {before_score:.3f}° -> {after_score:.3f}°; inspect fine/coarse and clean/noisy separately")
        if STOCHASTIC_SAVE_CANDIDATE and best_candidate is not None:
            candidate_path = RUN_DIR / f"stochastic_odometry_{STOCHASTIC_NOISE_SCHEDULE}_{STOCHASTIC_COARSE_MAX_DEG:g}deg.pt"
            torch.save(dict(model_state_dict=best_candidate, selected_step=best_step, fixed_score=best_score,
                            source_checkpoint=str(SOURCE_FM_CHECKPOINT), noise_config=noise_config_dict(),
                            schedule=STOCHASTIC_NOISE_SCHEDULE, target_intensity=STOCHASTIC_NOISE_INTENSITY), candidate_path)
            print("Saved diagnostic candidate:",candidate_path)
    finally:
        fm_model.load_state_dict(saved_weights, strict=True)
        set_coarse_angle(previous_coarse_deg)
        fm_model.train(saved_mode); fm_model.zero_grad(set_to_none=True)
        torch.random.set_rng_state(saved_cpu_rng)
        if saved_cuda_rng is not None: torch.cuda.set_rng_state_all(saved_cuda_rng)
        np.random.set_state(saved_numpy_rng); random.setstate(saved_python_rng)
        del saved_weights, small_base, fixed_cases
        if diagnostic_optimizer is not None: del diagnostic_optimizer
        gc.collect()
        if DEVICE.type=="cuda": torch.cuda.empty_cache()
    print("FM52 source weights and RNG restored. Main optimizer not touched.")
else:
    print("Stochastic check skipped; enable RUN_STOCHASTIC_CHECK before full training if desired.")


## 9. Stochastic diagnostic: rotation endpoint by noise condition, yaw, and fault regime

In [ ]:
if precheck_history:
    hist = pd.DataFrame(precheck_history)
    focus = hist[(hist.role=="active") & hist.key.str.startswith("lidar")]
    fig, axes = plt.subplots(2,2, figsize=(14,8), sharex=True)
    for r, regime in enumerate(("fine","coarse")):
        for c, noise in enumerate(("clean","corrupt")):
            ax=axes[r,c]
            for yaw, group in focus[(focus.regime==regime) & (focus.noise==noise)].groupby("yaw_deg"):
                group=group.groupby("step",as_index=False).mean(numeric_only=True).sort_values("step")
                ax.plot(group.step, group.mean_deg, label=f"Yaw {yaw:g}°")
            ax.set(title=f"{regime} | {noise}",xlabel="Diagnostic optimizer step",ylabel="Active LiDAR residual [deg]")
            ax.grid(alpha=.25); ax.legend(fontsize=8)
    fig.suptitle("FM53 stochastic: clean versus corrupted LiDAR", fontsize=14)
    fig.tight_layout(); plt.show()
    if precheck_final is not None:
        reduced = (precheck_final[precheck_final.role=="active"].groupby(["noise","regime","yaw_deg"],as_index=False)
                   .agg(before=("mean_deg_before","mean"),after=("mean_deg_after","mean")))
        reduced["delta_deg"]=reduced["after"]-reduced["before"]
        display(reduced.round(3))
else:
    print("No diagnostic history. Run the stochastic preflight above.")


## 10. Full training with noise mixture and independent clean/corrupted validation; exact FM53 resume

In [ ]:
if DEVICE.type == "cuda": torch.cuda.empty_cache()

In [ ]:
head_params = list(fm_model.head.parameters())
feature_params = [p for block in (fm_model.transformer_encoder,fm_model.pooler,fm_model.mlp_body) for p in block.parameters()]
assert len({id(p) for p in head_params+feature_params})==len(head_params+feature_params)
optimizer = torch.optim.AdamW([
    {"params":head_params,"lr":HEAD_LEARNING_RATE,"name":"velocity_head"},
    {"params":feature_params,"lr":BACKBONE_LEARNING_RATE,"name":"transformer_pooler_mlp"},
],weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="min",factor=LR_REDUCE_FACTOR,patience=LR_REDUCE_STEPS)


def run_fm53_epoch(loader, *, training, seed, regime="mixed", yaw_cap_deg=180.0, yaw_probability=1.0,
                   noise_mode="mixed", noise_intensity_value=1.0, max_batches=None):
    """A physical window has either both LiDAR streams clean or both corrupted.

    noise_mode='mixed' samples the configured window-level probability; 'clean'
    uses no corruption; 'corrupt' changes every LiDAR stream of every window.
    Fixed validation uses the same 1.0 noise intensity each epoch, not the curriculum.
    """
    if noise_mode not in ("clean","corrupt","mixed"): raise ValueError(noise_mode)
    fm_model.train(training)
    generator=augmentation_generator(seed)
    noise_generator=augmentation_generator(seed+400000)
    yaw_randomizer=make_yaw_randomizer(yaw_cap_deg,yaw_probability)
    noise_augmenter=odometry_noise_augmenter(noise_intensity_value)
    totals=defaultdict(float)
    examples,pending,pending_examples,updates=0,0,0,0
    started=time.perf_counter()
    if training: optimizer.zero_grad(set_to_none=True)

    def flush(n):
        for p in fm_model.parameters():
            if p.grad is not None: p.grad.mul_(1.0/n)
        torch.nn.utils.clip_grad_norm_(fm_model.parameters(),GRADIENT_CLIP_NORM)
        optimizer.step(); optimizer.zero_grad(set_to_none=True)

    with (torch.enable_grad() if training else torch.no_grad()):
        for i,cpu_batch in enumerate(loader):
            if max_batches is not None and i>=max_batches: break
            base=window_batch_to(cpu_batch,DEVICE,dtype=DTYPE)
            with torch.no_grad(): groups=weighted_regime_groups(base,generator,regime)
            batch_count=len(next(iter(base.current_calibration.values())).transform)
            for sampled_regime,subgroup in groups:
                with torch.no_grad():
                    calibrated=augment_original_window(subgroup,generator,regime=sampled_regime)
                    noise_parts=noise_subgroups(calibrated,noise_generator,ODOM_CORRUPTION_PROBABILITY) if noise_mode=="mixed" else [(noise_mode,calibrated)]
                for condition,sample in noise_parts:
                    with torch.no_grad():
                        augmented,_=corrupt_lidar_window(sample,noise_generator,noise_mode=condition,augmenter=noise_augmenter)
                    n=len(next(iter(augmented.current_calibration.values())).transform)
                    loss,metrics=evaluate_paired_objective(augmented,generator=generator,frame_randomizer=yaw_randomizer,
                                                           regime=sampled_regime,objective=FULL_OBJECTIVE,paired=TRAIN_PAIRED_VIEWS)
                    importance=regime_loss_multiplier(sampled_regime,regime,FULL_FINE_LOSS_WEIGHT,FULL_COARSE_LOSS_WEIGHT)
                    if training: (loss*(n*importance)).backward()
                    totals["loss_weighted"]+=float(loss.detach())*n*importance
                    totals["num_corrupt"]+=n if condition=="corrupt" else 0
                    totals[f"loss_{sampled_regime}"]+=float(loss.detach())*n
                    for key,val in metrics.items():
                        if np.isfinite(val):
                            totals[key]+=val*n
                            totals[f"{key}_weight"]+=n
            examples+=batch_count
            if training:
                pending+=1; pending_examples+=batch_count
                if pending==GRADIENT_ACCUMULATION_STEPS:
                    flush(pending_examples);updates+=1;pending=pending_examples=0
        if training and pending:
            flush(pending_examples);updates+=1
    if not examples: raise RuntimeError("Empty FM53 epoch")
    metrics={k:v/examples for k,v in totals.items() if not k.endswith("_weight")}
    for k in ("lidar_active","lidar_healthy","rotated","original","equivariance","endpoint","velocity"):
        if totals.get(f"{k}_weight",0): metrics[k]=totals[k]/totals[f"{k}_weight"]
    return {**metrics,"num_windows":examples,"optimizer_steps":updates,
            "seconds":time.perf_counter()-started,"yaw_cap":yaw_cap_deg,"noise_intensity":noise_intensity_value}

history_path=RUN_DIR/"fm53_full_history.csv"
history=pd.read_csv(history_path).to_dict("records") if history_path.is_file() else []
start_epoch,global_step,best_val=0,0,float("inf")
if RESUME_FULL_TRAINING:
    if not LAST_CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f"Cannot resume FM53 without its last.pt: {LAST_CHECKPOINT_PATH}")
    load_training_checkpoint(LAST_CHECKPOINT_PATH,model=fm_model,optimizer=optimizer,scheduler=scheduler,map_location=DEVICE,restore_rng_state=True)
    saved=torch.load(LAST_CHECKPOINT_PATH,map_location="cpu",weights_only=False)
    start_epoch,global_step=int(saved["epoch"]),int(saved["global_step"])
    info=saved.get("extra_state") or {}
    assert info.get("run_name")==RUN_NAME,"Cannot resume optimizer from a different run"
    assert info.get("noise_config")==noise_config_dict(),"Noise parameters changed; start a new run"
    assert info.get("full_noise_schedule")==FULL_NOISE_SCHEDULE and info.get("full_noise_intensity")==FULL_NOISE_INTENSITY,"Noise curriculum changed"
    assert info.get("full_ramp_start")==FULL_RAMP_START_INTENSITY and info.get("full_ramp_epochs")==FULL_RAMP_EPOCHS,"Ramp settings changed"
    assert info.get("noise_probability")==ODOM_CORRUPTION_PROBABILITY and info.get("val_noisy_weight")==VAL_NOISY_WEIGHT,"Corruption fractions changed"
    assert info.get("coarse_max_deg")==FULL_COARSE_MAX_DEG and info.get("train_regime")==FULL_TRAIN_REGIME,"Prior configuration changed"
    assert tuple(info.get("regime_weights",()))==(FULL_FINE_LOSS_WEIGHT,FULL_COARSE_LOSS_WEIGHT),"Regime weights changed"
    assert info.get("paired_views")==TRAIN_PAIRED_VIEWS and info.get("equivariance_weight")==EQUIVARIANCE_LOSS_WEIGHT,"Yaw objective changed"
    best_val=float(info.get("best_val",float("inf")))
    print("Resumed FM53",start_epoch,"optimizer steps",global_step)
else:
    if (history or LAST_CHECKPOINT_PATH.is_file()) and RUN_FULL_TRAINING:
        raise RuntimeError("Existing FM53 run detected. Enable RESUME_FULL_TRAINING or choose a new RUN_NAME")
    if history or LAST_CHECKPOINT_PATH.is_file():
        print("Existing FM53 checkpoints present; training disabled, weights for comparisons load from best.pt.")
    load_training_checkpoint(SOURCE_FM_CHECKPOINT,model=fm_model,map_location=DEVICE,restore_rng_state=False)
    print("Fresh FM53 model initialized from FM52 best:",SOURCE_FM_CHECKPOINT)

set_coarse_angle(FULL_COARSE_MAX_DEG)


def save_fm53_checkpoint(path,epoch,metrics):
    save_training_checkpoint(path,model=fm_model,optimizer=optimizer,scheduler=scheduler,epoch=epoch,global_step=global_step,metrics=metrics,
        extra_state=dict(run_name=RUN_NAME,best_val=best_val,source_checkpoint=str(SOURCE_FM_CHECKPOINT),
                         noise_config=noise_config_dict(),full_noise_schedule=FULL_NOISE_SCHEDULE,full_noise_intensity=FULL_NOISE_INTENSITY,
                         full_ramp_start=FULL_RAMP_START_INTENSITY,full_ramp_epochs=FULL_RAMP_EPOCHS,
                         noise_probability=ODOM_CORRUPTION_PROBABILITY,val_noisy_weight=VAL_NOISY_WEIGHT,
                         coarse_max_deg=FULL_COARSE_MAX_DEG,train_regime=FULL_TRAIN_REGIME,
                         regime_weights=(FULL_FINE_LOSS_WEIGHT,FULL_COARSE_LOSS_WEIGHT),
                         paired_views=TRAIN_PAIRED_VIEWS,equivariance_weight=EQUIVARIANCE_LOSS_WEIGHT))

if RUN_FULL_TRAINING:
    if REQUIRE_PRECHECK_PASS_FOR_FULL_TRAINING and STOCHASTIC_CHECK_PASSED is not True:
        raise RuntimeError("Stochastic check did not pass; inspect fixed clean/noisy cases")
    for epoch in range(start_epoch,TRAIN_EPOCHS):
        intensity=noise_intensity(epoch+1,schedule=FULL_NOISE_SCHEDULE,target=FULL_NOISE_INTENSITY,
                                   ramp_start=FULL_RAMP_START_INTENSITY,ramp_length=FULL_RAMP_EPOCHS)
        cap=training_yaw_cap(epoch+1)
        train_shuffle_generator.manual_seed(SEED+12001+epoch)
        train=run_fm53_epoch(train_loader,training=True,seed=TRAIN_AUG_SEED+100003*epoch,regime=FULL_TRAIN_REGIME,
                             yaw_cap_deg=cap,yaw_probability=FULL_YAW_PROBABILITY,noise_mode="mixed",noise_intensity_value=intensity,
                             max_batches=MAX_TRAIN_BATCHES_PER_EPOCH)
        # Independent model evaluation on clean and fully corrupted LiDAR data:
        # no corruption is applied to either IMU, and the same seeds are reused each epoch.
        val_clean=run_fm53_epoch(val_loader,training=False,seed=VAL_AUG_SEED,regime=FULL_TRAIN_REGIME,
                                  yaw_cap_deg=VAL_YAW_MAX_DEG,yaw_probability=VAL_YAW_PROBABILITY,noise_mode="clean",noise_intensity_value=1.0,
                                  max_batches=MAX_VAL_BATCHES_PER_EPOCH)
        val_corrupt=run_fm53_epoch(val_loader,training=False,seed=VAL_AUG_SEED,regime=FULL_TRAIN_REGIME,
                                    yaw_cap_deg=VAL_YAW_MAX_DEG,yaw_probability=VAL_YAW_PROBABILITY,noise_mode="corrupt",noise_intensity_value=1.0,
                                    max_batches=MAX_VAL_BATCHES_PER_EPOCH)
        val_score=float((1.0-VAL_NOISY_WEIGHT)*val_clean["loss_weighted"]+VAL_NOISY_WEIGHT*val_corrupt["loss_weighted"])
        scheduler.step(val_score)
        global_step+=int(train["optimizer_steps"])
        improved=val_score<best_val
        if improved:best_val=val_score
        row=dict(epoch=epoch+1,noise_intensity=intensity,train_loss=train["loss_weighted"],
                 val_clean=val_clean["loss_weighted"],val_corrupt=val_corrupt["loss_weighted"],val_loss=val_score,
                 clean_lidar_active=val_clean.get("lidar_active"),corrupt_lidar_active=val_corrupt.get("lidar_active"),
                 clean_lidar_healthy=val_clean.get("lidar_healthy"),corrupt_lidar_healthy=val_corrupt.get("lidar_healthy"),
                 train_equivariance=train.get("equivariance"),val_clean_equivariance=val_clean.get("equivariance"),
                 val_corrupt_equivariance=val_corrupt.get("equivariance"),head_lr=optimizer.param_groups[0]["lr"],
                 backbone_lr=optimizer.param_groups[1]["lr"])
        history.append(row);pd.DataFrame(history).to_csv(history_path,index=False)
        if improved:save_fm53_checkpoint(BEST_CHECKPOINT_PATH,epoch+1,{"loss":val_score,**row})
        save_fm53_checkpoint(LAST_CHECKPOINT_PATH,epoch+1,{"loss":val_score,**row})
        print(f"Epoch {epoch+1:3d}/{TRAIN_EPOCHS} noise={intensity:.2f} train={train['loss_weighted']:.3f} "
              f"val={val_score:.3f} clean={val_clean['loss_weighted']:.3f} corrupt={val_corrupt['loss_weighted']:.3f} "
              f"LiDAR(active)={val_clean.get('lidar_active',float('nan')):.1f}/{val_corrupt.get('lidar_active',float('nan')):.1f}° "
              f"LR={optimizer.param_groups[0]['lr']:.1e}/{optimizer.param_groups[1]['lr']:.1e}"+(" BEST" if improved else ""),flush=True)
else:
    print("Full training is disabled. Set RUN_FULL_TRAINING=True after the stochastic preflight.")


## 11. Train/validation histories and checkpoint selection

In [ ]:
if history:
    frame=pd.DataFrame(history)
    fig,axes=plt.subplots(1,3,figsize=(17,4.5))
    axes[0].plot(frame.epoch,frame.train_loss,label="Training (70/30)")
    axes[0].plot(frame.epoch,frame.val_clean,label="Val: clean")
    axes[0].plot(frame.epoch,frame.val_corrupt,label="Val: corrupted")
    axes[0].plot(frame.epoch,frame.val_loss,label="Val: weighted",linewidth=2)
    axes[0].set(title="FM objective",xlabel="Epoch",ylabel="Loss")
    for col,label in (("clean_lidar_active","Clean"),("corrupt_lidar_active","Corrupted")):
        if col in frame: axes[1].plot(frame.epoch,frame[col],label=label)
    axes[1].set(title="Active LiDAR endpoint error",xlabel="Epoch",ylabel="Degrees")
    for col,label in (("clean_lidar_healthy","Clean"),("corrupt_lidar_healthy","Corrupted")):
        if col in frame: axes[2].plot(frame.epoch,frame[col],label=label)
    axes[2].set(title="Healthy LiDAR drift",xlabel="Epoch",ylabel="Degrees")
    for ax in axes:ax.grid(alpha=.25);ax.legend()
    fig.tight_layout();plt.show()
    display(frame.tail(12).round(3))
else:
    print("No FM53 training history yet")

# Evaluate model from the best checkpoint first; otherwise use stochastic candidate.
RUN_FIXED_YAW_EVALUATION=False  # Enable after stochastic/full training; 2 noise levels x many yaw/angle cases.

def eval_checkpoint():
    if BEST_CHECKPOINT_PATH.is_file():return BEST_CHECKPOINT_PATH
    candidate=RUN_DIR/f"stochastic_odometry_{STOCHASTIC_NOISE_SCHEDULE}_{STOCHASTIC_COARSE_MAX_DEG:g}deg.pt"
    return candidate


def load_fm53_model(path):
    model=PooledCalibrationFlowModel(flow_config.backbone,**flow_config.model_kwargs()).to(DEVICE,dtype=DTYPE)
    # Stochastic candidates contain model_state_dict but no optimizer; official checkpoints are full.
    if path.name.startswith("stochastic_odometry_"):
        payload=torch.load(path,map_location="cpu",weights_only=False)
        model.load_state_dict(payload["model_state_dict"],strict=True)
    else:
        load_training_checkpoint(path,model=model,map_location=DEVICE,restore_rng_state=False)
    model.eval().requires_grad_(False)
    return model


## 12. Fixed yaw × angular-bin comparison with source FM52 (clean and corrupted LiDAR)

In [ ]:
# Optional expensive comparison, using the SAME physical fault/yaw cases in both
# measurement conditions and the SAME cases for FM52 and FM53.
EVAL_FAULT_KEYS=CALIBRATION_KEYS
EVAL_ANGLE_BINS=((0.,0.),(0.,5.),(5.,15.),(15.,30.),(30.,60.),(60.,90.),(90.,120.),(120.,150.),(150.,179.))
EVAL_YAW_DEGREES=(0.,45.,90.,135.,180.,225.,270.,315.)
EVAL_WINDOWS_PER_KEY=3
EVAL_SEED=VAL_AUG_SEED+950000
EVAL_OUTPUT=RUN_DIR/"fm53_fixed_yaw_angle_clean_noisy_comparison.csv"

@torch.no_grad()
def build_yaw_angular_cases():
    cases=[]
    indexes=np.unique(np.linspace(0,len(val_dataset)-1,EVAL_WINDOWS_PER_KEY).round().astype(int))
    noise_augmenter=odometry_noise_augmenter(1.0)
    for bin_index,(low,high) in enumerate(EVAL_ANGLE_BINS):
        for key_index,key in enumerate(EVAL_FAULT_KEYS):
            for item_index,window_index in enumerate(indexes):
                case_seed=EVAL_SEED+100000*bin_index+10000*key_index+item_index
                rng=augmentation_generator(case_seed)
                base=window_batch_to(collate_windows([val_dataset[int(window_index)]]),DEVICE,dtype=DTYPE)
                physical=physical_augmenter(base,generator=rng).augmented_window
                reference=physical.current_calibration
                sample=next(iter(reference.values())).transform
                xi={k:sample.new_zeros(1,6) for k in reference}
                tau={k:torch.zeros_like(s.time_offset) for k,s in reference.items()}
                axis=torch.randn(3,device=DEVICE,dtype=DTYPE,generator=rng)
                axis=axis/torch.linalg.vector_norm(axis).clamp_min(1e-12)
                angle=0.0 if low==high==0 else float(low+(high-low)*torch.rand((),device=DEVICE,generator=rng))
                xi[key][0,:3]=axis*np.deg2rad(angle)
                wrong=clone_with_current_calibration(physical,apply_prior_perturbation(reference,xi,tau))
                corrupted,_=corrupt_lidar_window(wrong,augmentation_generator(case_seed+50000),augmenter=noise_augmenter)
                for mode,window in (("clean",wrong),("corrupt",corrupted)):
                    for yaw in EVAL_YAW_DEGREES:
                        rotated,_=rotate_window_fixed_yaw(window,yaw)
                        cases.append(dict(window=window_batch_to(rotated,torch.device("cpu"),dtype=DTYPE),noise=mode,yaw_deg=yaw,
                                          bin="0°" if low==high==0. else f"{low:g}–{high:g}°",low=low,high=high,fault_key=key,
                                          initial_deg=angle,window_index=int(window_index)))
    return cases

@torch.no_grad()
def score_yaw_angular_cases(model,cases,label):
    rows=[]
    for case in cases:
        window=window_batch_to(case["window"],DEVICE,dtype=DTYPE)
        pred=model_predictions(model,window,steps=FM_INTEGRATION_STEPS)
        for key in CALIBRATION_KEYS:
            corrected=se3_exp(pred.predictions[key].delta_xi)@window.current_calibration[key].transform
            desired=window.targets[key].next_transform
            R=desired[:,:3,:3]@corrected[:,:3,:3].transpose(-1,-2)
            sine=.5*torch.linalg.vector_norm(torch.stack((R[:,2,1]-R[:,1,2],R[:,0,2]-R[:,2,0],R[:,1,0]-R[:,0,1]),-1),dim=-1)
            cosine=.5*(R.diagonal(dim1=-2,dim2=-1).sum(dim=-1)-1)
            error=float(torch.rad2deg(torch.atan2(sine,cosine)).mean())
            rows.append(dict(model=label,noise=case["noise"],bin=case["bin"],bin_low=case["low"],fault_key=case["fault_key"],
                             measured_key=key,active=(key==case["fault_key"]),yaw_deg=case["yaw_deg"],
                             window_index=case["window_index"],initial_deg=case["initial_deg"],final_deg=error,
                             improved=bool(error<case["initial_deg"]) if key==case["fault_key"] else False))
    return pd.DataFrame(rows)

if RUN_FIXED_YAW_EVALUATION and eval_checkpoint().is_file():
    fixed_cases=build_yaw_angular_cases()
    comparisons=[]
    for label,path in (("FM52 source",SOURCE_FM_CHECKPOINT),("FM53 candidate",eval_checkpoint())):
        model=load_fm53_model(path)
        comparisons.append(score_yaw_angular_cases(model,fixed_cases,label))
        del model;gc.collect()
        if DEVICE.type=="cuda":torch.cuda.empty_cache()
    yaw_angular_results=pd.concat(comparisons,ignore_index=True)
    yaw_angular_results.to_csv(EVAL_OUTPUT,index=False)
    display(yaw_angular_results[yaw_angular_results.active].groupby(["model","noise","yaw_deg","bin"],as_index=False)
            .agg(n=("final_deg","size"),final_mean_deg=("final_deg","mean"),improved_fraction=("improved","mean")).round(3))
else:
    print("Fixed yaw × angle comparison not run. Set RUN_FIXED_YAW_EVALUATION=True after saving a candidate.")


## 13. Clean/noisy yaw and angle comparison plots

In [ ]:
if "yaw_angular_results" in globals() and not yaw_angular_results.empty:
    data=yaw_angular_results
    fig,axes=plt.subplots(2,3,figsize=(16,9),sharex="col")
    for row,noise in enumerate(("clean","corrupt")):
        group=data[data.noise==noise]
        zero=group[(group.active)&(group.bin=="0°")]
        for model,part in zero.groupby("model"):
            p=part.groupby("yaw_deg",as_index=False).final_deg.mean().sort_values("yaw_deg")
            axes[row,0].plot(p.yaw_deg,p.final_deg,"-o",label=model)
        fine=group[(group.active)&(group.bin.isin(("0–5°","5–15°","15–30°","30–60°")))]
        for model,part in fine.groupby("model"):
            p=part.groupby("yaw_deg",as_index=False).final_deg.mean().sort_values("yaw_deg")
            axes[row,1].plot(p.yaw_deg,p.final_deg,"-o",label=model)
        coarse=group[(group.active)&(group.bin.isin(("90–120°","120–150°","150–179°")))]
        for model,part in coarse.groupby("model"):
            p=part.groupby("yaw_deg",as_index=False).final_deg.mean().sort_values("yaw_deg")
            axes[row,2].plot(p.yaw_deg,p.final_deg,"-o",label=model)
        for col,title in enumerate(("Zero injected fault","Fine injected angles","Coarse 90–179°")):
            axes[row,col].set(title=f"{noise} | {title}",xlabel="Common Z yaw [deg]",ylabel="Final active rotation [deg]")
            axes[row,col].grid(alpha=.25);axes[row,col].legend(fontsize=8)
    fig.tight_layout();plt.show()
    summary=(data.groupby(["model","noise","active","bin"],as_index=False)
             .agg(mean_final_deg=("final_deg","mean"),p90_final_deg=("final_deg",lambda x:x.quantile(.9)),
                  success=("improved","mean"),n=("final_deg","size")))
    display(summary.round(3))
else:
    print("No fixed angular comparison available")


## 14. Decision after the experiment

**Success:** FM53 decreases LiDAR endpoint errors on the **corrupted KAIST validation** distribution, keeps **clean KAIST** accuracy close to FM52, and shows transfer gains on a separately held-out New College evaluation without changing nominal calibration ground truth.

**If the stochastic check is flat:** compare fixed `clean` and `corrupt` fine/coarse curves. Try **rotation only** (`ODOM_ENABLE_TRANSLATION_SCALE=False`), then **translation scaling only** (`ODOM_ENABLE_ROTATION=False`). Avoid changing both at once until their effects are understood. Compare against `STOCHASTIC_NOISE_SCHEDULE="ramp"` with 100 updates to full severity if the fixed profile is too difficult. Keep separate `RUN_NAME`s for full runs with changed configs.

**If full training improves only the noise loss:** consider per-key LiDAR endpoint validation and physical **4–5 window rollouts**, since your numerical calibrator converges on that time horizon. This notebook does not train on chronologically persistent noise: correlated bias is resampled for each training window; a multi-window persistent-bias augmentation is a later extension.

**Scientific limitation:** New College's gyro/LiDAR residual includes motion, IMU, timing, ICP and mounting errors. The injected amplitudes are hypotheses, not ground-truth estimates of individual ICP error. Keep an independent final test segment to avoid tuning on New College indefinitely.